# Transcriptomics Analysis of *Daphnia magna* — Danube River Pollution

This notebook covers the full transcriptomics pipeline: quality control, exploratory data analysis, outlier removal, DESeq2 differential expression, WGCNA co-expression analysis, ortholog mapping, and presentation figure generation.

## 1. Setup and Package Loading

In [ ]:
# Load required packages
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(pheatmap)
  library(reshape2)
  library(scales)
  library(ggridges)
  library(patchwork)
  library(RColorBrewer)
  library(viridis)
  library(matrixStats)
  library(factoextra)
  library(FactoMineR)
  library(corrplot)
  library(ggrepel)
  library(DESeq2)
  library(WGCNA)
})


## 2. Colour Palettes and Data Loading

In [ ]:
# ── Colour palettes ───────────────────────────────────────────────────────────
# 13 levels: Control + D01-D12
SITE_COLS <- c(
  "Control" = "#888888",
  setNames(colorRampPalette(c("#4575b4","#74add1","#abd9e9",
                              "#e0f3f8","#fee090","#fdae61",
                              "#f46d43","#d73027","#a50026",
                              "#762a83","#9970ab","#c2a5cf"))(12),
           paste0("D", sprintf("%02d", 1:12)))
)

CONC_COLS <- c("Control" = "#4daf4a", "1x" = "#377eb8", "10x" = "#e41a1c")
CONC_LEVS <- c("Control", "1x", "10x")
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)


In [ ]:
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# ── The actual columns are: SampleID, REF, Site, Description
#    REF  = concentration level (Control / 1x / 10x)
#    Site = geographic site (Control / D01–D12)
colnames(ss) <- c("sample_id", "concentration", "site", "description")

ss$concentration <- factor(ss$concentration, levels = CONC_LEVS)
ss$site <- factor(ss$site, levels = c("Control", paste0("D", sprintf("%02d", 1:12))))

cat("Dimensions:", nrow(ss), "rows ×", ncol(ss), "cols\n")
cat("Unique sites:", paste(levels(ss$site), collapse = ", "), "\n")
cat("Concentration levels:", paste(levels(ss$concentration), collapse = ", "), "\n")

In [ ]:
tab <- as.data.frame(table(Site = ss$site, Conc = ss$concentration))
tab <- tab[tab$Freq > 0, ]
spread_tab <- reshape(tab, idvar = "Site", timevar = "Conc", direction = "wide")
colnames(spread_tab) <- gsub("Freq.", "", colnames(spread_tab))
spread_tab[order(spread_tab$Site), ]

In [ ]:
ggplot(ss, aes(x = site, fill = concentration)) +
  geom_bar(position = "dodge", colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = CONC_COLS) +
  labs(
    title   = "Samples per site and concentration",
    subtitle = "6 biological replicates per group; A = 1x, B = 10x",
    x = "Site (upstream D01 → downstream D12)",
    y = "Number of samples",
    fill = "Concentration"
  ) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 3. Chemical Data Exploration

In [ ]:
chem <- read.delim("water_chemicals.tsv",stringsAsFactors = FALSE, check.names = FALSE)

# Columns: ChemName, CAS, D01 … D12
cat("Dimensions:", nrow(chem), "chemicals ×", ncol(chem), "columns\n")
site_cols <- paste0("D", sprintf("%02d", 1:12))

# Convert site columns to numeric
for (sc in site_cols) chem[[sc]] <- as.numeric(chem[[sc]])

head(chem[, c("ChemName", "CAS", site_cols[1:4])], 8)

In [ ]:
detect_counts <- colSums(chem[, site_cols] > 0, na.rm = TRUE)
detect_df <- data.frame(
  site       = factor(names(detect_counts), levels = site_cols),
  n_detected = as.integer(detect_counts)
)

ggplot(detect_df, aes(x = site, y = n_detected, fill = site)) +
  geom_col(colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = SITE_COLS[site_cols], guide = "none") +
  geom_text(aes(label = n_detected), vjust = -0.4, size = 3.5) +
  labs(
    title   = "Number of chemicals detected above LOD per site",
    subtitle = "Zero = below detection limit (not included)",
    x = "Site", y = "Chemicals detected (n)"
  ) +
  ylim(0, max(detect_df$n_detected) * 1.1) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
chem$n_sites_detected <- rowSums(chem[, site_cols] > 0, na.rm = TRUE)

top20_chem <- chem[order(chem$n_sites_detected, decreasing = TRUE), ][1:20, ]

ggplot(top20_chem, aes(x = reorder(ChemName, n_sites_detected), y = n_sites_detected)) +
  geom_col(fill = "#377eb8", colour = "white", linewidth = 0.3) +
  coord_flip() +
  labs(
    title = "Top 20 most prevalent chemicals",
    subtitle = "By number of sites where concentration > 0",
    x = NULL, y = "Number of sites (out of 12)"
  ) +
  theme_minimal(base_size = 11)

In [ ]:
library(ggplot2)

# Load data
chem <- read.table("water_chemicals.tsv", header=T, sep="\t", row.names=1, check.names=F)
chem_values <- chem[, -1]

# Create chem_var data frame with all metrics
chem_var <- data.frame(
  Chemical = row.names(chem_values),
  Variance = apply(chem_values, 1, var),
  Mean = apply(chem_values, 1, mean),
  CV = apply(chem_values, 1, sd) / apply(chem_values, 1, mean)
)

# Remove chemicals with zero mean
chem_var_filtered <- chem_var[chem_var$Mean > 0, ]

# Sort by CV
chem_var_filtered <- chem_var_filtered[order(chem_var_filtered$CV, decreasing = TRUE), ]

# Plot top 20
top20_cv <- head(chem_var_filtered, 20)
top20_cv$Chemical <- factor(top20_cv$Chemical, levels = rev(top20_cv$Chemical))

ggplot(top20_cv, aes(x = CV, y = Chemical)) +
  geom_bar(stat = "identity", fill = "coral") +
  labs(
    title = "Top 20 most relatively variable chemicals across sites",
    subtitle = "By coefficient of variation (SD/Mean) across 12 sites",
    x = "Coefficient of Variation",
    y = ""
  ) +
  theme_minimal()

In [ ]:
library(pheatmap)

# Read the data
chem <- read.table("water_chemicals.tsv", header=T, sep="\t", row.names=1, check.names=F)

# Remove CAS column, keep only site columns
chem_mat <- as.matrix(chem[, -1])

# Filter to chemicals detected in at least 1 site
detected <- rowSums(chem_mat > 0) > 0
chem_mat <- chem_mat[detected, ]

# Log10 transform with pseudocount
log_mat <- log10(chem_mat + 0.001)
log_mat[is.infinite(log_mat)] <- NA

pheatmap(
  log_mat,
  main         = "Chemical concentrations (log10 ug/L, 1x level)",
  color        = colorRampPalette(c("white", "#fee8c8", "#fc8d59", "#b30000"))(100),
  na_col       = "#e8e8e8",
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  fontsize_row = 7,
  fontsize_col = 10,
  angle_col    = 45,
  border_color = NA
)

In [ ]:
library(corrplot)

# Read the data
chem <- read.table("water_chemicals.tsv", header=T, sep="\t", row.names=1, check.names=F)

# Remove CAS column, keep only site columns
chem_site <- chem[, -1]

# Make sure everything is numeric
chem_site <- as.data.frame(lapply(chem_site, as.numeric))

# Spearman correlation between sites
site_cor <- cor(chem_site, use = "pairwise.complete.obs", method = "spearman")

corrplot(
  site_cor,
  method      = "color",
  type        = "upper",
  col         = colorRampPalette(c("#4575b4", "white", "#d73027"))(100),
  addCoef.col = "black",
  number.cex  = 0.7,
  tl.cex      = 0.9,
  title       = "Spearman correlation between sites (chemical profiles)",
  mar         = c(0, 0, 2, 0)
)

## 4. RNA-seq Exploratory Data Analysis

In [ ]:
chem <- read.table("water_chemicals.tsv", header=T, sep="\t", row.names=1, check.names=F)
rna <- read.csv("rna_vst_counts.csv",row.names = 1, check.names = FALSE)

cat("Dimensions:", nrow(rna), "genes ×", ncol(rna), "samples\n")

# Align sample sheet to RNA columns
ss_rna <- ss[ss$sample_id %in% colnames(rna), ]
cat("Samples matched to sample sheet:", nrow(ss_rna), "\n")

In [ ]:
library(ggplot2)
library(ggridges)

set.seed(42)
samp_show <- sample(colnames(rna), 30)

rna_sub <- rna[, samp_show]
rna_sub$gene <- rownames(rna_sub)
rna_long <- reshape2::melt(rna_sub, id.vars = "gene",
                           variable.name = "sample_id", value.name = "vst")
rna_long <- merge(rna_long, ss, by = "sample_id")
rna_long$sample_id <- reorder(rna_long$sample_id, as.numeric(rna_long$site))

ggplot(rna_long, aes(x = vst, y = sample_id, fill = concentration)) +
  geom_density_ridges(alpha = 0.75, scale = 1.2,
                      colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = c("Control" = "grey", "1x" = "steelblue", "10x" = "coral")) +
  theme_minimal(base_size = 11)

In [ ]:
library(matrixStats)
library(FactoMineR)
library(ggplot2)

# Use top 5000 most variable genes
gene_vars <- rowVars(as.matrix(rna))
top5k     <- order(gene_vars, decreasing = TRUE)[1:5000]
rna_top   <- t(rna[top5k, ])  # samples × genes

pca_rna <- PCA(rna_top, ncp = 10, graph = FALSE)

pca_df <- as.data.frame(pca_rna$ind$coord)
pca_df$sample_id <- rownames(pca_df)
pca_df <- merge(pca_df, ss, by = "sample_id")
pca_df$concentration <- factor(pca_df$concentration, levels = c("Control", "1x", "10x"))

pct <- round(pca_rna$eig[1:4, 2], 1)

ggplot(pca_df, aes(x = Dim.1, y = Dim.2, colour = site, shape = concentration)) +
  geom_point(size = 3.5, alpha = 0.9) +
  scale_colour_brewer(palette = "Paired") +
  scale_shape_manual(values = c("Control" = 16, "1x" = 17, "10x" = 15)) +
  labs(
    title    = "RNA-seq PCA — PC1 vs PC2",
    subtitle = "Coloured by site; shape = concentration",
    x = paste0("PC1 (", pct[1], "%)"),
    y = paste0("PC2 (", pct[2], "%)"),
    colour = "Site", shape = "Concentration"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "right")

In [ ]:
library(ggplot2)

# Load raw counts and calculate library sizes
raw <- read.table("rna_raw_counts.csv", header=T, sep=",", row.names=1, check.names=F)
lib_sizes <- colSums(raw)

# Load sample sheet
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Create a data frame for plotting
lib_df <- data.frame(
  sample_id = names(lib_sizes),
  lib_size = lib_sizes
)

# Merge with sample sheet
lib_df <- merge(lib_df, ss, by.x = "sample_id", by.y = "SampleID")

# Sort by library size
lib_df <- lib_df[order(lib_df$lib_size), ]
lib_df$sample_id <- factor(lib_df$sample_id, levels = lib_df$sample_id)

# Flag low-quality samples
lib_df$quality <- ifelse(lib_df$lib_size < 1e6, "Low", "OK")

ggplot(lib_df, aes(x = sample_id, y = lib_size / 1e6, fill = quality)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = c("Low" = "red", "OK" = "steelblue")) +
  geom_hline(yintercept = 1, linetype = "dashed", colour = "red", linewidth = 0.5) +
  labs(
    title = "Library sizes per sample",
    subtitle = "Red bars and dashed line indicate samples below 1 million reads",
    x = NULL,
    y = "Total reads (millions)",
    fill = "Quality"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 5),
    legend.position = "top"
  )

In [ ]:
# Find samples with PC1 > 25 (the right-side outliers)
outliers <- pca_df$sample_id[pca_df$Dim.1 > 25]
print(outliers)

# Check their metadata
ss[ss$sample_id %in% outliers, ]

In [ ]:
library(ggplot2)

# Load raw counts and calculate genes detected
raw <- read.table("rna_raw_counts.csv", header=T, sep=",", row.names=1, check.names=F)
genes_detected <- colSums(raw > 0)

# Define outliers (from PCA analysis)
outliers <- c("CK09", "CK12", "D01A4", "D01A6", "D02A3", "D02A4", "D03A4", "D03B5",
              "D04A6", "D04B3", "D04B6", "D05A1", "D05A2", "D05B4", "D08A5", "D08A6",
              "D08B2", "D08B6", "D09A3", "D09B6", "D11B6", "D12A3", "D12B5")

# Create data frame
genes_df <- data.frame(
  sample_id = names(genes_detected),
  genes = genes_detected
)

# Label outliers vs main cluster
genes_df$group <- ifelse(genes_df$sample_id %in% outliers, "PCA Outliers", "Main Cluster")

ggplot(genes_df, aes(x = group, y = genes, fill = group)) +
  geom_boxplot(alpha = 0.7, outlier.shape = NA) +
  geom_jitter(width = 0.2, size = 1.5, alpha = 0.5) +
  scale_fill_manual(values = c("PCA Outliers" = "coral", "Main Cluster" = "steelblue")) +
  labs(
    title = "Genes detected per sample",
    subtitle = "Comparing PCA outlier group vs main cluster",
    x = NULL,
    y = "Number of genes with count > 0",
    fill = NULL
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

In [ ]:
library(ggplot2)

ggplot(pca_df, aes(x = Dim.1, y = Dim.2, colour = concentration, shape = site)) +
  geom_point(size = 3.5, alpha = 0.9) +
  scale_colour_manual(values = CONC_COLS) +
  labs(
    title    = "RNA-seq PCA — PC1 vs PC2",
    subtitle = "Coloured by concentration — does 10x separate from 1x?",
    x = paste0("PC1 (", pct[1], "%)"),
    y = paste0("PC2 (", pct[2], "%)"),
    colour = "Concentration"
  ) +
  guides(shape = "none") +
  theme_minimal(base_size = 13)

In [ ]:
library(ggplot2)
library(patchwork)

p13 <- ggplot(pca_df, aes(Dim.1, Dim.3, colour = site, shape = concentration)) +
  geom_point(size = 2.5, alpha = 0.85) +
  scale_colour_brewer(palette = "Paired", guide = "none") +
  scale_shape_manual(values = c("Control" = 16, "1x" = 17, "10x" = 15)) +
  labs(x = paste0("PC1 (", pct[1], "%)"),
       y = paste0("PC3 (", pct[3], "%)"), shape = "Concentration") +
  theme_minimal(base_size = 11)

p23 <- ggplot(pca_df, aes(Dim.2, Dim.3, colour = concentration)) +
  geom_point(size = 2.5, alpha = 0.85) +
  scale_colour_manual(values = c("Control" = "grey", "1x" = "steelblue", "10x" = "coral")) +
  labs(x = paste0("PC2 (", pct[2], "%)"),
       y = paste0("PC3 (", pct[3], "%)"), colour = "Concentration") +
  theme_minimal(base_size = 11)

# Display side by side
p13 | p23

In [ ]:
var_df <- data.frame(
  PC      = factor(paste0("PC", 1:10), levels = paste0("PC", 1:10)),
  var_pct = pca_rna$eig[1:10, 2]
)

ggplot(var_df, aes(x = PC, y = var_pct)) +
  geom_col(fill = "#377eb8", colour = "white", linewidth = 0.3) +
  geom_line(aes(group = 1), colour = "#e41a1c", linewidth = 0.8) +
  geom_point(colour = "#e41a1c", size = 2.5) +
  labs(title = "RNA-seq — variance explained per PC",
       x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)

In [ ]:
library(matrixStats)

# Load data
rna <- read.table("rna_vst_counts.csv", header=T, sep=",", row.names=1, check.names=F)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Get top 5000 most variable genes
gene_vars <- rowVars(as.matrix(rna))
top5k <- order(gene_vars, decreasing = TRUE)[1:5000]
rna_top <- t(rna[top5k, ])

# Run PCA
pca <- prcomp(rna_top, scale. = TRUE)

# Make a data frame with PC scores and metadata
pca_df <- data.frame(pca$x[, 1:10])
pca_df$sample_id <- rownames(pca_df)
pca_df <- merge(pca_df, ss, by.x = "sample_id", by.y = "SampleID")

# R-squared for site vs concentration on each PC
summary(lm(PC1 ~ Site, data = pca_df))$r.squared
summary(lm(PC1 ~ REF, data = pca_df))$r.squared
summary(lm(PC2 ~ Site, data = pca_df))$r.squared
summary(lm(PC2 ~ REF, data = pca_df))$r.squared

In [ ]:
library(pheatmap)

# Load data
rna <- read.table("rna_vst_counts.csv", header=T, sep=",", row.names=1, check.names=F)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Check column names to use the right ones
# colnames(ss)

# Pearson correlation on VST counts
rna_cor <- cor(rna, method = "pearson")

# Build annotation data frame
ann_col <- data.frame(
  site = ss$Site,
  concentration = ss$REF,
  row.names = ss$SampleID
)

# Keep only samples that exist in the correlation matrix
ann_col <- ann_col[rownames(ann_col) %in% colnames(rna_cor), ]

# Order by site then concentration
ann_col <- ann_col[order(ann_col$site, ann_col$concentration), ]

# Reorder correlation matrix to match
rna_cor_ord <- rna_cor[rownames(ann_col), rownames(ann_col)]

pheatmap(
  rna_cor_ord,
  main              = "Sample-sample Pearson correlation (VST counts)",
  color             = colorRampPalette(c("#4575b4", "white", "#d73027"))(100),
  annotation_col    = ann_col,
  annotation_row    = ann_col,
  show_rownames     = FALSE,
  show_colnames     = FALSE,
  border_color      = NA,
  fontsize          = 8
)

In [ ]:
library(ggplot2)
library(matrixStats)
library(reshape2)

# Load data
rna <- read.table("rna_vst_counts.csv", header=T, sep=",", row.names=1, check.names=F)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Calculate gene variances
gene_vars <- rowVars(as.matrix(rna))
names(gene_vars) <- rownames(rna)

# Get top 20 most variable genes
top20_genes <- names(sort(gene_vars, decreasing = TRUE))[1:20]
rna_top20 <- rna[top20_genes, ]
rna_top20$gene <- rownames(rna_top20)

rna_top20_long <- melt(rna_top20, id.vars = "gene",
                       variable.name = "sample_id", value.name = "vst")

# Merge - use the correct column name from sample sheet
rna_top20_long <- merge(rna_top20_long, ss, by.x = "sample_id", by.y = "SampleID")

ggplot(rna_top20_long, aes(x = Site, y = vst, fill = REF)) +
  geom_boxplot(outlier.size = 0.5, linewidth = 0.3) +
  scale_fill_manual(values = c("Control" = "grey", "1x" = "steelblue", "10x" = "coral")) +
  facet_wrap(~ gene, scales = "free_y", ncol = 5) +
  labs(title = "Top 20 most variable genes across all samples",
       x = NULL, y = "VST count", fill = "Concentration") +
  theme_minimal(base_size = 9) +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 6),
        strip.text = element_text(size = 7))

In [ ]:
# PCA on chemical profiles across sites (transpose: sites = rows)
library(ggplot2)
library(FactoMineR)

# Load data
chem <- read.table("water_chemicals.tsv", header=T, sep="\t", row.names=1, check.names=F)
chem_mat_raw <- as.matrix(chem[, -1])  # Remove CAS column
chem_mat_raw[is.na(chem_mat_raw)] <- 0

# Transpose so sites are rows
chem_pca_input <- t(chem_mat_raw)

pca_chem <- PCA(chem_pca_input, graph = FALSE)
pca_chem_df <- as.data.frame(pca_chem$ind$coord)
pca_chem_df$site <- rownames(pca_chem_df)

pct_c <- round(pca_chem$eig[1:2, 2], 1)

ggplot(pca_chem_df, aes(x = Dim.1, y = Dim.2, colour = site, label = site)) +
  geom_point(size = 5) +
  geom_text(nudge_y = 0.5, size = 3.5) +
  scale_colour_brewer(palette = "Paired", guide = "none") +
  labs(
    title = "Chemical profile PCA — one point per site",
    subtitle = "Sites close together have similar chemical mixtures",
    x = paste0("PC1 (", pct_c[1], "%)"),
    y = paste0("PC2 (", pct_c[2], "%)")
  ) +
  theme_minimal(base_size = 13)

## 5. Quality Control Summary

In [ ]:
data.frame(
  Check = c(
    "Sample sheet parsed correctly",
    "RNA samples match sample sheet",
    "Genes in RNA matrix",
    "Chemicals detected (at least 1 site)",
    "Controls present",
    "Replicates per treatment group"
  ),
  Result = c(
    paste0(nrow(ss), " samples: 6 Control, 72 × 1x, 72 × 10x"),
    paste0(nrow(ss_rna), " / ", nrow(ss), " matched"),
    paste0(format(nrow(rna), big.mark = ","), " genes"),
    paste0(sum(rowSums(as.data.frame(lapply(chem[, -1], as.numeric)) > 0) > 0), " / ", nrow(chem), " chemicals"),
    "6 control samples (CK07–CK12)",
    "6 per site × concentration"
  )
)

## 6. Outlier Removal

### Outlier Removal

Based on the previous PCA and quality control plots (library size, genes detected), several samples were identified as potential outliers. Removing these can improve the quality and interpretability of downstream differential expression analysis. We will proceed by filtering out these identified samples from both the VST counts data and the sample sheet.



In [ ]:
library(dplyr)

# Load data again to ensure we have the original ones
rna <- read.table("rna_vst_counts.csv", header=T, sep=",", row.names=1, check.names=F)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Outlier samples identified in previous steps (e.g., cell `urXy8IAPOQsR`)
outliers <- c("CK09", "CK12", "D01A4", "D01A6", "D02A3", "D02A4", "D03A4", "D03B5",
              "D04A6", "D04B3", "D04B6", "D05A1", "D05A2", "D05B4", "D08A5", "D08A6",
              "D08B2", "D08B6", "D09A3", "D09B6", "D11B6", "D12A3", "D12B5")

cat("Original RNA dimensions:", nrow(rna), "genes x", ncol(rna), "samples\n")
cat("Original sample sheet dimensions:", nrow(ss), "rows x", ncol(ss), "cols\n")
cat("Number of outliers to remove:", length(outliers), "\n\n")

# Remove outlier samples from the RNA VST counts matrix
rna_filtered <- rna[, !colnames(rna) %in% outliers]

# Remove outlier samples from the sample sheet
ss_filtered <- ss[!ss$SampleID %in% outliers, ]

cat("Filtered RNA dimensions:", nrow(rna_filtered), "genes x", ncol(rna_filtered), "samples\n")
cat("Filtered sample sheet dimensions:", nrow(ss_filtered), "rows x", ncol(ss_filtered), "cols\n")

# Display the head of the filtered sample sheet to confirm
head(ss_filtered)

### Visual Confirmation of Outliers

To ensure the robustness of our outlier identification, we'll visualize the PCA again, explicitly marking the samples previously flagged as outliers. This allows for a clear visual inspection of their position relative to the main cluster and the threshold used for their removal.

In [ ]:
library(ggplot2)
library(FactoMineR)
library(matrixStats)

# Outlier samples (from cell urXy8IAPOQsR) - define early as it's used before ss is filtered
outliers <- c("CK09", "CK12", "D01A4", "D01A6", "D02A3", "D02A4", "D03A4", "D03B5",
              "D04A6", "D04B3", "D04B6", "D05A1", "D05A2", "D05B4", "D08A5", "D08A6",
              "D08B2", "D08B6", "D09A3", "D09B6", "D11B6", "D12A3", "D12B5")

# Reload data to ensure original state for PCA calculation
rna <- read.table("rna_vst_counts.csv", header=T, sep=",", row.names=1, check.names=F)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Apply column renaming and factor conversion to ss (replicate logic from MoBB7z1BNBWh)
colnames(ss) <- c("sample_id", "concentration", "site", "description")
# Ensure CONC_LEVS are available or hardcoded for consistency
CONC_LEVS <- c("Control", "1x", "10x")
ss$concentration <- factor(ss$concentration, levels = CONC_LEVS)
ss$site <- factor(ss$site, levels = c("Control", paste0("D", sprintf("%02d", 1:12))))

# Define SITE_COLS for consistent coloring (copied from X9vP4HSwMEyX for robustness)
SITE_COLS <- c(
  "Control" = "#888888",
  setNames(colorRampPalette(c("#4575b4","#74add1","#abd9e9",
                              "#e0f3f8","#fee090","#fdae61",
                              "#f46d43","#d73027","#a50026",
                              "#762a83","#9970ab","#c2a5cf"))(12),
           paste0("D", sprintf("%02d", 1:12)))
)

# Use top 5000 most variable genes (from cell jFoRWDdvOFoG)
gene_vars <- rowVars(as.matrix(rna))
top5k     <- order(gene_vars, decreasing = TRUE)[1:5000]
rna_top   <- t(rna[top5k, ])  # samples × genes

pca_rna <- PCA(rna_top, ncp = 10, graph = FALSE)

pca_df <- as.data.frame(pca_rna$ind$coord)
pca_df$sample_id <- rownames(pca_df)
# Now merge using "sample_id" as the common column after renaming ss columns
pca_df <- merge(pca_df, ss, by = "sample_id") # No need for by.x, by.y if column names match

pct <- round(pca_rna$eig[1:4, 2], 1)

pca_df$is_outlier <- ifelse(pca_df$sample_id %in% outliers, "Outlier", "Main Cluster")

ggplot(pca_df, aes(x = Dim.1, y = Dim.2, colour = site, shape = is_outlier)) +
  geom_point(size = 3.5, alpha = 0.9) +
  scale_colour_manual(values = SITE_COLS) + # Use the custom SITE_COLS palette
  scale_shape_manual(values = c("Main Cluster" = 16, "Outlier" = 8)) + # Use a distinct shape for outliers
  geom_vline(xintercept = 25, linetype = "dashed", color = "red", linewidth = 0.8) + # Mark the threshold
  labs(
    title    = "RNA-seq PCA — PC1 vs PC2 (Outliers Highlighted)",
    subtitle = paste0("Samples with PC1 > 25 identified as outliers. Variance explained: PC1 (", pct[1], "%), PC2 (", pct[2], "%)\nNote: This plot uses the *original* data for PCA, before outlier removal. (12 samples removed due to missing values.)"),
    x = paste0("PC1 (", pct[1], "%)"),
    y = paste0("PC2 (", pct[2], "%)"),
    colour = "Site", shape = "Sample Group"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "right")

In [ ]:
outlier_libs <- lib_df$lib_size[lib_df$sample_id %in% outliers]
main_libs    <- lib_df$lib_size[!lib_df$sample_id %in% outliers]
wilcox.test(outlier_libs, main_libs)
# plus: min(outlier_libs) — is anyone below 1M?

In [ ]:
pct_zero <- colMeans(raw == 0)
summary(pct_zero[outliers])
summary(pct_zero[setdiff(colnames(raw), outliers)])

In [ ]:
# Top genes contributing to PC1
loadings <- pca_rna$var$coord[, 1]
top_pc1_genes <- names(sort(abs(loadings), decreasing = TRUE))[1:30]
print(top_pc1_genes)

In [ ]:
table(ss$concentration[ss$sample_id %in% outliers])
# If mostly "10x" → biology
# If random mix including Control → technical

In [ ]:
###############################################################################
# Pre-DESeq2 checks:
# 1. Which 2 samples are below 1M reads? (definite technical removals)
# 2. What biology do the top PC1 genes represent? (outlier interpretation)
###############################################################################

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
})

# ------------------------------------------------------------------------------
# CHECK 1: Samples below 1M reads
# ------------------------------------------------------------------------------

raw <- read.csv("rna_raw_counts.csv", row.names = 1, check.names = FALSE)
lib_sizes <- colSums(raw)

cat("=== Library size summary ===\n")
cat("Min library size :", format(min(lib_sizes), big.mark = ","), "\n")
cat("Median           :", format(median(lib_sizes), big.mark = ","), "\n")
cat("Max              :", format(max(lib_sizes), big.mark = ","), "\n\n")

low_lib_samples <- names(lib_sizes[lib_sizes < 1e6])
cat("Samples below 1M reads (to REMOVE):\n")
print(data.frame(
  sample = low_lib_samples,
  library_size = lib_sizes[low_lib_samples]
))

# Also check: are these low-lib samples among the 23 PCA outliers?
outliers <- c("CK09", "CK12", "D01A4", "D01A6", "D02A3", "D02A4", "D03A4", "D03B5",
              "D04A6", "D04B3", "D04B6", "D05A1", "D05A2", "D05B4", "D08A5", "D08A6",
              "D08B2", "D08B6", "D09A3", "D09B6", "D11B6", "D12A3", "D12B5")

cat("\nOverlap between low-library and PCA outliers:\n")
print(intersect(low_lib_samples, outliers))

cat("\n=> REMOVE these samples before DESeq2:",
    paste(low_lib_samples, collapse = ", "), "\n")

# Save the exclusion list for DESeq2
saveRDS(low_lib_samples, "samples_to_exclude.rds")

# ------------------------------------------------------------------------------
# CHECK 2: Map top PC1 genes to human + fly orthologs
# ------------------------------------------------------------------------------
# The mapping files supplied by the course:
#   rna_dma_to_hsa_gn_mappings.tsv   — D. magna -> Human gene symbols
#   rna_dma_to_hsa_mappings.tsv      — D. magna -> Human gene names
#   rna_dma_to_dme_mappings.tsv      — D. magna -> Drosophila FlyBase IDs
#   rna_dma_to_dpx_mappings.tsv      — D. magna -> D. pulex

top_pc1_genes <- c(
  "Dapma7bEVm015878","Dapma7bEVm012638","Dapma7bEVm008883","Dapma7bEVm018818",
  "Dapma7bEVm025865","Dapma7bEVm016781","Dapma7bEVm021992","Dapma7bEVm009361",
  "Dapma7bEVm026891","Dapma7bEVm008625","Dapma7bEVm002306","Dapma7bEVm017416",
  "Dapma7bEVm004761","Dapma7bEVm025941","Dapma7bEVm015890","Dapma7bEVm012877",
  "Dapma7bEVm025747","Dapma7bEVm029389","Dapma7bEVm026361","Dapma7bEVm026857",
  "Dapma7bEVm026215","Dapma7bEVm016417","Dapma7bEVm016233","Dapma7bEVm008757",
  "Dapma7bEVm008315","Dapma7bEVm030454","Dapma7bEVm025881","Dapma7bEVm027853",
  "Dapma7bEVm007235","Dapma7bEVm028275"
)

# Helper to load a mapping file robustly (handles CSV vs TSV, column name variants)
load_mapping <- function(path) {
  if (!file.exists(path)) {
    cat("File not found:", path, "\n")
    return(NULL)
  }
  # Try TSV first, fall back to CSV
  df <- tryCatch(
    read.delim(path, stringsAsFactors = FALSE, check.names = FALSE),
    error = function(e) read.csv(path, stringsAsFactors = FALSE, check.names = FALSE)
  )
  cat("\nLoaded:", path, "| rows:", nrow(df), "| cols:", ncol(df), "\n")
  cat("Column names:", paste(colnames(df), collapse = " | "), "\n")
  cat("First row   :", paste(unlist(df[1, ]), collapse = " | "), "\n")
  df
}

cat("\n\n=== Mapping top 30 PC1 genes to orthologs ===\n")

hsa_gn   <- load_mapping("rna_dma_to_hsa_gn_mappings.tsv")
hsa_full <- load_mapping("rna_dma_to_hsa_mappings.tsv")
dme      <- load_mapping("rna_dma_to_dme_mappings.tsv")
dpx      <- load_mapping("rna_dma_to_dpx_mappings.tsv")

# Function to extract ortholog info for a list of D. magna genes.
# Assumes the first column of each mapping file is the D. magna gene ID.
lookup_orthologs <- function(mapping_df, dma_ids, label) {
  if (is.null(mapping_df)) return(NULL)
  dma_col <- colnames(mapping_df)[1]
  matched <- mapping_df[mapping_df[[dma_col]] %in% dma_ids, ]
  cat(sprintf("\n--- %s: %d/%d genes matched ---\n",
              label, length(unique(matched[[dma_col]])), length(dma_ids)))
  if (nrow(matched) > 0) print(head(matched, 50))
  matched
}

hsa_matched <- lookup_orthologs(hsa_gn,   top_pc1_genes, "Human gene symbols")
hsa_full_m  <- lookup_orthologs(hsa_full, top_pc1_genes, "Human (full/ensembl)")
dme_matched <- lookup_orthologs(dme,      top_pc1_genes, "Drosophila FlyBase")
dpx_matched <- lookup_orthologs(dpx,      top_pc1_genes, "D. pulex")

# Save a consolidated summary so you can screenshot / paste into slides
if (!is.null(hsa_matched) && nrow(hsa_matched) > 0) {
  write.csv(hsa_matched, "top30_pc1_genes_human_orthologs.csv", row.names = FALSE)
  cat("\nSaved: top30_pc1_genes_human_orthologs.csv\n")
}

# ------------------------------------------------------------------------------
# Interpretation guide: keywords to look for in the human gene names/symbols
# ------------------------------------------------------------------------------
cat("\n=== What to look for in the human symbols ===\n")
cat("Detox / xenobiotic response (STRONG evidence of pollution biology):\n")
cat("  CYP* (cytochrome P450)   — phase I metabolism\n")
cat("  GST* (glutathione S-transferase) — phase II detox\n")
cat("  UGT* (UDP-glucuronosyltransferase) — phase II detox\n")
cat("  ABCB*, ABCC*, ABCG*     — drug efflux pumps\n")
cat("  MT*, MT1*, MT2*         — metallothioneins (metal binding)\n")
cat("\nStress response (supports pollution biology):\n")
cat("  HSP*, HSPA*, HSPB*      — heat shock proteins\n")
cat("  SOD*, CAT, GPX*         — oxidative stress\n")
cat("  FOXO*, NRF2/NFE2L2      — stress transcription factors\n")
cat("\nImmune / inflammation:\n")
cat("  TLR*, IL*, NFKB*        — innate immunity\n")
cat("\nWarning signs — suggests TECHNICAL rather than biological:\n")
cat("  RPL*, RPS*              — ribosomal (housekeeping)\n")
cat("  MT-ND*, MT-CO*, MT-CYB  — mitochondrial-encoded (MT- prefix)\n")
cat("  ACTB, GAPDH, TUB*       — classic housekeeping\n")

cat("\n\n=== DONE ===\n")
cat("Next: look at the printed human symbols and match them to the categories above.\n")

### Final Outlier Decision Summary

Based on the comprehensive quality control analysis:

1.  **Technical Exclusion (Low Library Size):** 2 samples (D03A6, D03B3) that fell below the 1 million read threshold were identified and **excluded**. These represent clear technical quality issues.
2.  **PCA Outlier Cluster (Control Samples):** From the additional cluster of 23 samples separated on PC1 (>25 score, accounting for 22% variance), 2 control samples (CK09, CK12) were identified. As there is no biological explanation for control samples to cluster separately, these two samples were **excluded** to prevent confounding technical artifacts with biological signals.
3.  **PCA Outlier Cluster (Treatment Samples):** The remaining 21 treatment samples from this PCA outlier cluster were **retained**. Investigation of the top PC1-loading genes revealed an enrichment for mitochondrial-encoded transcripts (e.g., MT-ND4). While this might suggest a technical influence, 26 out of the 30 top PC1 genes lacked human orthologs, making it difficult to definitively rule out a Daphnia-specific biological response. Therefore, to be cautious, these samples will be included in the primary analysis, but their impact will be assessed through sensitivity analysis (comparing results with and without these samples) in the final report.

This refined sample set will now be used for downstream differential expression analysis.

## 7. Differential Expression Analysis (DESeq2)

In [ ]:
###############################################################################
# DAY 2 — Differential expression with DESeq2
#
# Design: ~ group, where group = paste(site, concentration)
# This lets us pull any contrast we want cleanly, including per-site contrasts.
#
# Contrasts extracted:
#   (A) Pooled 10x vs Control  — strongest "pollution signature"
#   (B) Pooled 1x  vs Control  — environmentally-realistic signature
#   (C) Per-site 10x vs Control  — site-specific toxicity (12 contrasts)
#   (D) Per-site 1x  vs Control  — site-specific low-dose (12 contrasts)
###############################################################################

suppressPackageStartupMessages({
  library(DESeq2)
  library(dplyr)
  library(ggplot2)
})

dir.create("deseq2_results", showWarnings = FALSE)

# ---- Load data ---------------------------------------------------------------
raw <- read.csv("rna_raw_counts.csv", row.names = 1, check.names = FALSE)
ss  <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Your sample sheet column names: SampleID, REF, Site, Description
# Normalize to lowercase for consistency
colnames(ss) <- c("sample_id", "concentration", "site", "description")

# ---- Remove low-library samples ---------------------------------------------
exclude <- c("D03A6", "D03B3")
cat("Excluding low-library samples:", paste(exclude, collapse = ", "), "\n")

raw <- raw[, !colnames(raw) %in% exclude]
ss  <- ss[!ss$sample_id %in% exclude, ]

cat("Samples remaining:", ncol(raw), "\n")
stopifnot(ncol(raw) == 148)

# ---- Round raw counts to integers -------------------------------------------
# Salmon gives decimal counts; DESeq2 requires integers
raw_int <- round(as.matrix(raw))

# ---- Align sample sheet to count matrix -------------------------------------
ss <- ss[match(colnames(raw_int), ss$sample_id), ]
stopifnot(all(colnames(raw_int) == ss$sample_id))

# ---- Build the 'group' factor -----------------------------------------------
# Combine site and concentration into one factor (reference = Control_Control)
ss$group <- paste(ss$site, ss$concentration, sep = "_")
ss$group <- factor(ss$group)

# Make sure "Control_Control" is the reference level
if ("Control_Control" %in% levels(ss$group)) {
  ss$group <- relevel(ss$group, ref = "Control_Control")
}

cat("\nGroup breakdown:\n")
print(table(ss$group))

# ---- Build DESeq2 object and run --------------------------------------------
dds <- DESeqDataSetFromMatrix(
  countData = raw_int,
  colData   = ss,
  design    = ~ group
)

# Pre-filter: keep genes with >=10 counts in at least 6 samples (size of smallest group)
keep <- rowSums(counts(dds) >= 10) >= 6
cat("\nGenes before filtering:", nrow(dds), "\n")
dds <- dds[keep, ]
cat("Genes after filtering :", nrow(dds), "\n\n")

cat("Running DESeq2... (this takes ~2-5 minutes)\n")
dds <- DESeq(dds, parallel = FALSE)
cat("DESeq2 done.\n\n")

saveRDS(dds, "deseq2_results/dds.rds")

# ---- Helper: extract & annotate a contrast ----------------------------------
get_contrast <- function(dds, numerator_groups, denominator_groups,
                         label, padj_thresh = 0.05, lfc_thresh = 1) {
  # numerator_groups / denominator_groups: character vectors of group levels to
  #   average over. For pooled contrasts, pass all sites at that concentration.
  #   For per-site contrasts, pass single group names.

  # Build the contrast vector using "listValues"-style averaging
  # We'll use results() with contrast = list(numerator_groups, denominator_groups)
  # but resultsNames() gives coefficients, so we need to map groups to coefs.
  # Simplest: use contrast = c("group", num_single, den_single) for 1v1,
  # and the listed form for pooled comparisons.

  if (length(numerator_groups) == 1 && length(denominator_groups) == 1) {
    res <- results(dds, contrast = c("group",
                                     numerator_groups, denominator_groups),
                   alpha = padj_thresh)
  } else {
    # Pooled contrast: average of coefficients
    res_names <- resultsNames(dds)
    # Convert group names to their coefficient names (prefix "group")
    num_coefs <- paste0("group", numerator_groups)
    den_coefs <- paste0("group", denominator_groups)
    # Remove any that aren't in resultsNames (e.g., the reference)
    num_coefs <- intersect(num_coefs, res_names)
    den_coefs <- intersect(den_coefs, res_names)

    res <- results(dds,
                   contrast = list(num_coefs, den_coefs),
                   listValues = c(1/length(num_coefs), -1/length(den_coefs)),
                   alpha = padj_thresh)
  }

  df <- as.data.frame(res)
  df$gene_id <- rownames(df)
  df <- df[, c("gene_id", "baseMean", "log2FoldChange", "lfcSE",
               "stat", "pvalue", "padj")]
  df <- df[order(df$padj), ]

  n_sig <- sum(df$padj < padj_thresh & abs(df$log2FoldChange) > lfc_thresh,
               na.rm = TRUE)
  n_up  <- sum(df$padj < padj_thresh & df$log2FoldChange > lfc_thresh,
               na.rm = TRUE)
  n_dn  <- sum(df$padj < padj_thresh & df$log2FoldChange < -lfc_thresh,
               na.rm = TRUE)

  cat(sprintf("  %-40s  DEGs (padj<%.2f, |lfc|>%.0f): %d  (up:%d, down:%d)\n",
              label, padj_thresh, lfc_thresh, n_sig, n_up, n_dn))

  list(table = df, n_sig = n_sig, n_up = n_up, n_down = n_dn)
}

# ---- Contrast A: Pooled 10x vs Control --------------------------------------
cat("=== Contrast A: Pooled 10x vs Control ===\n")
sites <- paste0("D", sprintf("%02d", 1:12))
groups_10x <- paste0(sites, "_10x")
groups_10x <- intersect(groups_10x, levels(ss$group))

res_A <- get_contrast(dds,
                      numerator_groups   = groups_10x,
                      denominator_groups = "Control_Control",
                      label = "Pooled 10x vs Control")

write.csv(res_A$table, "deseq2_results/A_pooled_10x_vs_Control.csv",
          row.names = FALSE)

# ---- Contrast B: Pooled 1x vs Control ---------------------------------------
cat("\n=== Contrast B: Pooled 1x vs Control ===\n")
groups_1x <- paste0(sites, "_1x")
groups_1x <- intersect(groups_1x, levels(ss$group))

res_B <- get_contrast(dds,
                      numerator_groups   = groups_1x,
                      denominator_groups = "Control_Control",
                      label = "Pooled 1x vs Control")

write.csv(res_B$table, "deseq2_results/B_pooled_1x_vs_Control.csv",
          row.names = FALSE)

# ---- Contrast C: Per-site 10x vs Control ------------------------------------
cat("\n=== Contrast C: Per-site 10x vs Control ===\n")
per_site_10x <- data.frame(site = character(), n_deg = integer(),
                           n_up = integer(), n_down = integer(),
                           stringsAsFactors = FALSE)

for (s in sites) {
  grp <- paste0(s, "_10x")
  if (!grp %in% levels(ss$group)) next

  r <- get_contrast(dds,
                    numerator_groups   = grp,
                    denominator_groups = "Control_Control",
                    label = sprintf("%s 10x vs Control", s))

  write.csv(r$table,
            sprintf("deseq2_results/C_%s_10x_vs_Control.csv", s),
            row.names = FALSE)

  per_site_10x <- rbind(per_site_10x,
                        data.frame(site = s, n_deg = r$n_sig,
                                   n_up = r$n_up, n_down = r$n_down))
}

write.csv(per_site_10x, "deseq2_results/per_site_10x_summary.csv",
          row.names = FALSE)

# ---- Contrast D: Per-site 1x vs Control -------------------------------------
cat("\n=== Contrast D: Per-site 1x vs Control ===\n")
per_site_1x <- data.frame(site = character(), n_deg = integer(),
                          n_up = integer(), n_down = integer(),
                          stringsAsFactors = FALSE)

for (s in sites) {
  grp <- paste0(s, "_1x")
  if (!grp %in% levels(ss$group)) next

  r <- get_contrast(dds,
                    numerator_groups   = grp,
                    denominator_groups = "Control_Control",
                    label = sprintf("%s 1x vs Control", s))

  write.csv(r$table,
            sprintf("deseq2_results/D_%s_1x_vs_Control.csv", s),
            row.names = FALSE)

  per_site_1x <- rbind(per_site_1x,
                       data.frame(site = s, n_deg = r$n_sig,
                                  n_up = r$n_up, n_down = r$n_down))
}

write.csv(per_site_1x, "deseq2_results/per_site_1x_summary.csv",
          row.names = FALSE)

# ---- Summary bar plot: DEGs per site ----------------------------------------
summary_df <- rbind(
  data.frame(per_site_10x, dose = "10x"),
  data.frame(per_site_1x,  dose = "1x")
)
summary_df$dose <- factor(summary_df$dose, levels = c("1x", "10x"))
summary_df$site <- factor(summary_df$site, levels = sites)

p_summary <- ggplot(summary_df, aes(site, n_deg, fill = dose)) +
  geom_col(position = "dodge", colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = c("1x" = "#377eb8", "10x" = "#e41a1c")) +
  labs(
    title = "Differentially expressed genes per site",
    subtitle = "padj < 0.05 and |log2FC| > 1, each site vs shared Control pool",
    x = "Site (upstream D01 → downstream D12)",
    y = "Number of DEGs",
    fill = "Dose"
  ) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggsave("deseq2_results/degs_per_site.png", p_summary,
       width = 9, height = 5, dpi = 150)

# ---- Save a top-hits table from the pooled 10x contrast ---------------------
top_hits <- res_A$table %>%
  filter(!is.na(padj), padj < 0.05, abs(log2FoldChange) > 1) %>%
  arrange(padj) %>%
  head(50)

write.csv(top_hits, "deseq2_results/top50_pooled_10x_DEGs.csv",
          row.names = FALSE)

cat("\n=== DAY 2 DONE ===\n")
cat("Results saved in deseq2_results/\n")
cat("Key files:\n")
cat("  - A_pooled_10x_vs_Control.csv   (your main DEG table)\n")
cat("  - B_pooled_1x_vs_Control.csv    (low-dose signature)\n")
cat("  - C_*_10x_vs_Control.csv        (per-site 10x, 12 files)\n")
cat("  - D_*_1x_vs_Control.csv         (per-site 1x, 12 files)\n")
cat("  - per_site_10x_summary.csv      (DEG counts by site)\n")
cat("  - per_site_1x_summary.csv\n")
cat("  - top50_pooled_10x_DEGs.csv     (pretty table for slides)\n")
cat("  - degs_per_site.png             (summary bar plot)\n")
cat("  - dds.rds                       (full DESeq2 object)\n")

In [ ]:
resultsNames(dds)

In [ ]:
###############################################################################
# MINIMAL FIX for day 2
# Just redefine get_contrast() with the correct coefficient naming pattern,
# then re-run the contrast blocks.
#
# Your resultsNames(dds) uses format: "group_<level>_vs_Control_Control"
# not "group<level>" as I assumed previously.
###############################################################################

suppressPackageStartupMessages({
  library(DESeq2)
  library(dplyr)
  library(ggplot2)
})

dir.create("deseq2_results", showWarnings = FALSE)

# Reload dds if you lost the session
if (!exists("dds")) dds <- readRDS("deseq2_results/dds.rds")

# Rebuild ss if needed
if (!exists("ss")) {
  ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)
  colnames(ss) <- c("sample_id", "concentration", "site", "description")
  ss <- ss[!ss$sample_id %in% c("D03A6", "D03B3"), ]
  ss$group <- factor(paste(ss$site, ss$concentration, sep = "_"))
  ss$group <- relevel(ss$group, ref = "Control_Control")
}

sites <- paste0("D", sprintf("%02d", 1:12))

# Helper: map a group level to its DESeq2 coefficient name
# Levels look like "D01_10x"; coefficient names look like "group_D01_10x_vs_Control_Control"
group_to_coef <- function(group_level) {
  paste0("group_", group_level, "_vs_Control_Control")
}

# ---- FIXED get_contrast -----------------------------------------------------
get_contrast <- function(dds, numerator_groups, denominator_groups,
                         label, padj_thresh = 0.05, lfc_thresh = 1) {

  res_names <- resultsNames(dds)

  # Case 1: 1-vs-1 contrast (per-site) — simple built-in form works fine
  if (length(numerator_groups) == 1 && length(denominator_groups) == 1) {
    res <- results(dds,
                   contrast = c("group", numerator_groups, denominator_groups),
                   alpha = padj_thresh)

  # Case 2: pooled contrast against the reference level (Control_Control)
  # Build a numeric contrast vector by hand; each numerator coefficient gets 1/n
  } else if (length(denominator_groups) == 1 &&
             denominator_groups == "Control_Control") {

    num_coefs <- group_to_coef(numerator_groups)
    num_coefs <- intersect(num_coefs, res_names)

    if (length(num_coefs) == 0) {
      stop("No matching coefficients for numerator: ",
           paste(numerator_groups, collapse = ", "),
           "\nAvailable names: ", paste(res_names, collapse = ", "))
    }

    contrast_vec <- rep(0, length(res_names))
    names(contrast_vec) <- res_names
    contrast_vec[num_coefs] <- 1 / length(num_coefs)

    res <- results(dds, contrast = contrast_vec, alpha = padj_thresh)

  } else {
    stop("Unsupported contrast configuration.")
  }

  df <- as.data.frame(res)
  df$gene_id <- rownames(df)
  df <- df[, c("gene_id", "baseMean", "log2FoldChange", "lfcSE",
               "stat", "pvalue", "padj")]
  df <- df[order(df$padj), ]

  n_sig <- sum(df$padj < padj_thresh & abs(df$log2FoldChange) > lfc_thresh,
               na.rm = TRUE)
  n_up  <- sum(df$padj < padj_thresh & df$log2FoldChange > lfc_thresh,
               na.rm = TRUE)
  n_dn  <- sum(df$padj < padj_thresh & df$log2FoldChange < -lfc_thresh,
               na.rm = TRUE)

  cat(sprintf("  %-40s  DEGs (padj<%.2f, |lfc|>%.0f): %d  (up:%d, down:%d)\n",
              label, padj_thresh, lfc_thresh, n_sig, n_up, n_dn))

  list(table = df, n_sig = n_sig, n_up = n_up, n_down = n_dn)
}

# ---- Contrast A: Pooled 10x vs Control --------------------------------------
cat("=== Contrast A: Pooled 10x vs Control ===\n")
groups_10x <- intersect(paste0(sites, "_10x"), levels(ss$group))
res_A <- get_contrast(dds, groups_10x, "Control_Control",
                      "Pooled 10x vs Control")
write.csv(res_A$table, "deseq2_results/A_pooled_10x_vs_Control.csv",
          row.names = FALSE)

# ---- Contrast B: Pooled 1x vs Control ---------------------------------------
cat("\n=== Contrast B: Pooled 1x vs Control ===\n")
groups_1x <- intersect(paste0(sites, "_1x"), levels(ss$group))
res_B <- get_contrast(dds, groups_1x, "Control_Control",
                      "Pooled 1x vs Control")
write.csv(res_B$table, "deseq2_results/B_pooled_1x_vs_Control.csv",
          row.names = FALSE)

# ---- Contrast C: Per-site 10x vs Control ------------------------------------
cat("\n=== Contrast C: Per-site 10x vs Control ===\n")
per_site_10x <- data.frame()
for (s in sites) {
  grp <- paste0(s, "_10x")
  if (!grp %in% levels(ss$group)) next
  r <- get_contrast(dds, grp, "Control_Control",
                    sprintf("%s 10x vs Control", s))
  write.csv(r$table, sprintf("deseq2_results/C_%s_10x_vs_Control.csv", s),
            row.names = FALSE)
  per_site_10x <- rbind(per_site_10x,
                        data.frame(site = s, n_deg = r$n_sig,
                                   n_up = r$n_up, n_down = r$n_down))
}
write.csv(per_site_10x, "deseq2_results/per_site_10x_summary.csv",
          row.names = FALSE)

# ---- Contrast D: Per-site 1x vs Control -------------------------------------
cat("\n=== Contrast D: Per-site 1x vs Control ===\n")
per_site_1x <- data.frame()
for (s in sites) {
  grp <- paste0(s, "_1x")
  if (!grp %in% levels(ss$group)) next
  r <- get_contrast(dds, grp, "Control_Control",
                    sprintf("%s 1x vs Control", s))
  write.csv(r$table, sprintf("deseq2_results/D_%s_1x_vs_Control.csv", s),
            row.names = FALSE)
  per_site_1x <- rbind(per_site_1x,
                       data.frame(site = s, n_deg = r$n_sig,
                                  n_up = r$n_up, n_down = r$n_down))
}
write.csv(per_site_1x, "deseq2_results/per_site_1x_summary.csv",
          row.names = FALSE)

# ---- Summary bar plot ------------------------------------------------------
summary_df <- rbind(
  data.frame(per_site_10x, dose = "10x"),
  data.frame(per_site_1x,  dose = "1x")
)
summary_df$dose <- factor(summary_df$dose, levels = c("1x", "10x"))
summary_df$site <- factor(summary_df$site, levels = sites)

p_summary <- ggplot(summary_df, aes(site, n_deg, fill = dose)) +
  geom_col(position = "dodge", colour = "white", linewidth = 0.3) +
  scale_fill_manual(values = c("1x" = "#377eb8", "10x" = "#e41a1c")) +
  labs(title = "Differentially expressed genes per site",
       subtitle = "padj < 0.05 and |log2FC| > 1, each site vs shared Control",
       x = "Site (upstream D01 → downstream D12)",
       y = "Number of DEGs", fill = "Dose") +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggsave("deseq2_results/degs_per_site.png", p_summary,
       width = 9, height = 5, dpi = 150)

# ---- Top hits table --------------------------------------------------------
top_hits <- res_A$table %>%
  filter(!is.na(padj), padj < 0.05, abs(log2FoldChange) > 1) %>%
  arrange(padj) %>%
  head(50)
write.csv(top_hits, "deseq2_results/top50_pooled_10x_DEGs.csv",
          row.names = FALSE)

cat("\n=== DAY 2 DONE ===\n")

In [ ]:
###############################################################################
# Verify raw counts column names match sample sheet
# The slide deck warned they may not — let's see for real
###############################################################################

raw <- read.csv("rna_raw_counts.csv", row.names = 1, check.names = FALSE)
vst <- read.csv("rna_vst_counts.csv", row.names = 1, check.names = FALSE)
ss  <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

sheet_ids <- ss$SampleID  # per your earlier check, column is called SampleID

cat("=== Column name head-to-head ===\n")
cat("\nFirst 10 RAW column names:\n")
print(head(colnames(raw), 10))

cat("\nFirst 10 VST column names:\n")
print(head(colnames(vst), 10))

cat("\nFirst 10 SAMPLE SHEET IDs:\n")
print(head(sheet_ids, 10))

cat("\n=== Exact-match counts ===\n")
raw_match <- length(intersect(colnames(raw), sheet_ids))
vst_match <- length(intersect(colnames(vst), sheet_ids))
cat("Raw columns matching sample sheet:", raw_match, "/", length(sheet_ids), "\n")
cat("VST columns matching sample sheet:", vst_match, "/", length(sheet_ids), "\n")

# Are the raw and vst column sets identical?
cat("\nRaw and VST have identical columns:",
    setequal(colnames(raw), colnames(vst)), "\n")

cat("\n=== What's in RAW but NOT in sample sheet ===\n")
unmatched_raw <- setdiff(colnames(raw), sheet_ids)
cat(length(unmatched_raw), "columns unmatched\n")
if (length(unmatched_raw) > 0) {
  cat("First 20 examples:\n")
  print(head(unmatched_raw, 20))
}

cat("\n=== What's in sample sheet but NOT in RAW ===\n")
unmatched_ss <- setdiff(sheet_ids, colnames(raw))
cat(length(unmatched_ss), "sample sheet entries unmatched\n")
if (length(unmatched_ss) > 0) {
  cat("First 20 examples:\n")
  print(head(unmatched_ss, 20))
}

cat("\n=== Verdict ===\n")
if (raw_match == length(sheet_ids)) {
  cat("[OK] All", length(sheet_ids), "sample sheet entries match raw counts exactly.\n")
  cat("The slide deck warning does NOT apply to this file. DESeq2 ran on correct samples.\n")
} else {
  cat("[MISMATCH] ", length(sheet_ids) - raw_match,
      "sample sheet entries do NOT match raw counts.\n")
  cat("Need to fix before DESeq2 — your previous run may have dropped or mislabelled samples.\n")
}

In [ ]:
# Check that the counts stored in dds are integers
counts_in_dds <- counts(dds)

cat("Class of counts:", class(counts_in_dds[1, 1]), "\n")
cat("Are all counts integer values?",
    all(counts_in_dds == round(counts_in_dds)), "\n")
cat("Any decimal remainders?",
    any(counts_in_dds != floor(counts_in_dds)), "\n")
cat("Max decimal remainder:",
    max(abs(counts_in_dds - round(counts_in_dds))), "\n")

# Compare: do the counts in dds match what we'd get by rounding raw?
raw_check <- read.csv("rna_raw_counts.csv", row.names = 1, check.names = FALSE)
raw_check <- raw_check[, !colnames(raw_check) %in% c("D03A6", "D03B3")]
raw_check <- round(as.matrix(raw_check))

# Subset raw_check to same genes as dds (DESeq2 filtered some out)
common_genes <- intersect(rownames(raw_check), rownames(counts_in_dds))
raw_check_sub <- raw_check[common_genes, colnames(counts_in_dds)]
dds_sub      <- counts_in_dds[common_genes, ]

cat("Do rounded raw values match counts stored in dds?",
    all(raw_check_sub == dds_sub), "\n")

## 8. DEG Diagnostics and Volcano Plots

In [ ]:
###############################################################################
# DAY 2.5 — DEG diagnostics
# Volcano plots for key contrasts + overlap analysis across sites
###############################################################################

suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(ggrepel)
  library(reshape2)
})

dir.create("deseq2_results/diagnostics", showWarnings = FALSE)

# ---- Volcano helper ---------------------------------------------------------
make_volcano <- function(df, title, padj_thresh = 0.05, lfc_thresh = 1,
                         label_top_n = 10) {
  df <- df[!is.na(df$padj) & !is.na(df$log2FoldChange), ]
  df$neg_log_padj <- -log10(pmax(df$padj, 1e-300))

  df$category <- "NS"
  df$category[df$padj < padj_thresh & df$log2FoldChange >  lfc_thresh] <- "Up"
  df$category[df$padj < padj_thresh & df$log2FoldChange < -lfc_thresh] <- "Down"
  df$category <- factor(df$category, levels = c("NS", "Down", "Up"))

  df$label <- ""
  top_sig <- df[df$category != "NS", ]
  top_sig <- top_sig[order(top_sig$padj), ]
  top_sig <- head(top_sig, label_top_n)
  df$label[match(top_sig$gene_id, df$gene_id)] <- top_sig$gene_id

  n_up   <- sum(df$category == "Up")
  n_down <- sum(df$category == "Down")

  ggplot(df, aes(log2FoldChange, neg_log_padj, colour = category)) +
    geom_point(size = 1.2, alpha = 0.6) +
    geom_hline(yintercept = -log10(padj_thresh),
               linetype = "dashed", colour = "grey40", linewidth = 0.4) +
    geom_vline(xintercept = c(-lfc_thresh, lfc_thresh),
               linetype = "dashed", colour = "grey40", linewidth = 0.4) +
    scale_colour_manual(values = c("NS" = "grey70",
                                   "Down" = "#377eb8",
                                   "Up" = "#e41a1c")) +
    ggrepel::geom_text_repel(aes(label = label), size = 3,
                             max.overlaps = 20, box.padding = 0.4,
                             show.legend = FALSE) +
    labs(title = title,
         subtitle = sprintf("padj < %.2f, |log2FC| > %d  ->  Up: %d, Down: %d",
                            padj_thresh, lfc_thresh, n_up, n_down),
         x = "log2 Fold Change", y = "-log10(padj)", colour = NULL) +
    theme_minimal(base_size = 12) +
    theme(legend.position = "top")
}

# ---- Volcanoes for the 5 most informative contrasts -------------------------
contrasts_to_plot <- list(
  list(file = "A_pooled_10x_vs_Control.csv",
       title = "Pooled 10x vs Control (all sites)"),
  list(file = "B_pooled_1x_vs_Control.csv",
       title = "Pooled 1x vs Control (all sites)"),
  list(file = "C_D11_10x_vs_Control.csv",
       title = "D11 10x vs Control"),
  list(file = "D_D11_1x_vs_Control.csv",
       title = "D11 1x vs Control"),
  list(file = "D_D12_1x_vs_Control.csv",
       title = "D12 1x vs Control")
)

for (cc in contrasts_to_plot) {
  path <- file.path("deseq2_results", cc$file)
  if (!file.exists(path)) {
    cat("Skipping missing:", path, "\n"); next
  }
  df <- read.csv(path, stringsAsFactors = FALSE)
  p  <- make_volcano(df, cc$title)
  out <- file.path("deseq2_results/diagnostics",
                   sub("\\.csv$", "_volcano.png", cc$file))
  ggsave(out, p, width = 7.5, height = 6, dpi = 150)
  cat("Saved:", out, "\n")
}

# ---- DEG overlap analysis --------------------------------------------------
get_deg_ids <- function(path, padj_thresh = 0.05, lfc_thresh = 1) {
  if (!file.exists(path)) return(character(0))
  df <- read.csv(path, stringsAsFactors = FALSE)
  df <- df[!is.na(df$padj) & !is.na(df$log2FoldChange), ]
  df$gene_id[df$padj < padj_thresh & abs(df$log2FoldChange) > lfc_thresh]
}

sites <- paste0("D", sprintf("%02d", 1:12))

deg_sets <- list()
for (s in sites) {
  f10 <- file.path("deseq2_results", sprintf("C_%s_10x_vs_Control.csv", s))
  f1  <- file.path("deseq2_results", sprintf("D_%s_1x_vs_Control.csv",  s))
  deg_sets[[sprintf("%s_10x", s)]] <- get_deg_ids(f10)
  deg_sets[[sprintf("%s_1x",  s)]] <- get_deg_ids(f1)
}

jaccard <- function(a, b) {
  if (length(a) == 0 || length(b) == 0) return(0)
  length(intersect(a, b)) / length(union(a, b))
}

set_names <- names(deg_sets)
J <- matrix(0, nrow = length(set_names), ncol = length(set_names),
            dimnames = list(set_names, set_names))
for (i in seq_along(set_names)) {
  for (j in seq_along(set_names)) {
    J[i, j] <- jaccard(deg_sets[[i]], deg_sets[[j]])
  }
}

write.csv(round(J, 3), "deseq2_results/diagnostics/deg_overlap_jaccard.csv")

J_df <- melt(J, varnames = c("Set1", "Set2"), value.name = "jaccard")

p_overlap <- ggplot(J_df, aes(Set1, Set2, fill = jaccard)) +
  geom_tile(colour = "white", linewidth = 0.3) +
  scale_fill_gradient(low = "#f7fbff", high = "#08306b", limits = c(0, 1)) +
  labs(title = "DEG overlap between site-dose conditions",
       subtitle = "Jaccard similarity: |intersect| / |union| of DEG sets",
       x = NULL, y = NULL, fill = "Jaccard") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        panel.grid = element_blank())

ggsave("deseq2_results/diagnostics/deg_overlap_heatmap.png",
       p_overlap, width = 10, height = 9, dpi = 150)

# ---- Core DEG analysis -----------------------------------------------------
all_degs <- unlist(deg_sets)
gene_freq <- sort(table(all_degs), decreasing = TRUE)

core_df <- data.frame(
  gene_id = names(gene_freq),
  n_conditions = as.integer(gene_freq)
)

n_core_5  <- sum(core_df$n_conditions >= 5)
n_core_10 <- sum(core_df$n_conditions >= 10)
n_core_15 <- sum(core_df$n_conditions >= 15)

cat("\n=== Core DEG analysis ===\n")
cat("Total unique DEGs across any condition:", nrow(core_df), "\n")
cat("DEG in >= 5 conditions :", n_core_5, "\n")
cat("DEG in >= 10 conditions:", n_core_10, "\n")
cat("DEG in >= 15 conditions:", n_core_15, "\n")

write.csv(core_df,
          "deseq2_results/diagnostics/deg_frequency_across_conditions.csv",
          row.names = FALSE)

p_freq <- ggplot(core_df, aes(n_conditions)) +
  geom_histogram(binwidth = 1, fill = "#4a6fa5", colour = "white") +
  labs(title = "Generality of DEGs across site-dose conditions",
       subtitle = sprintf("n = %d unique DEGs across 24 conditions (12 sites x 2 doses)",
                          nrow(core_df)),
       x = "Number of site-dose conditions where gene is DEG",
       y = "Number of genes") +
  theme_minimal(base_size = 12)

ggsave("deseq2_results/diagnostics/deg_frequency_histogram.png",
       p_freq, width = 7, height = 5, dpi = 150)

cat("\n=== Done ===\n")
cat("Outputs in deseq2_results/diagnostics/:\n")
cat("  - 5 volcano plots for key contrasts\n")
cat("  - deg_overlap_heatmap.png / .csv  (Jaccard similarity)\n")
cat("  - deg_frequency_histogram.png\n")
cat("  - deg_frequency_across_conditions.csv\n")


## 9. Ortholog Mapping

In [ ]:
###############################################################################
# DR METHOD !!!!!!!! CORRECT ortholog mapping pipeline (prof's recommended approach)
#
# Step 1: Anchor Dma -> D. pulex (dpx) — evolutionarily closest, most reliable
# Step 2: Use dpx as hub — only keep Dma genes with a dpx counterpart
# Step 3: Cross-reference: check dme and hsa through the same anchor
# Step 4: Resolve 1:many by picking ONE ortholog per Dma gene
#
# Key difference from the previous script:
#   - Filters Dma genes to those with a confident dpx ortholog FIRST
#   - Resolves 1:many by picking the first (primary) ortholog, documenting this
#   - Reports coverage at each step for transparency
###############################################################################

suppressPackageStartupMessages({
  library(dplyr)
})

# ---- Load and repair mapping files -----------------------------------------
# (Same fix as before — first "header" is actually data)
load_map <- function(path) {
  df <- read.delim(path, stringsAsFactors = FALSE, check.names = FALSE)
  first_col_name <- colnames(df)[1]
  if (grepl("^Dapma", first_col_name)) {
    header_row <- data.frame(dma = first_col_name,
                             ortho = colnames(df)[2],
                             stringsAsFactors = FALSE)
    colnames(df) <- c("dma", "ortho")
    df <- rbind(header_row, df)
  } else {
    colnames(df) <- c("dma", "ortho")
  }
  df$ortho[df$ortho == ""] <- NA
  df
}

dpx_map <- load_map("rna_dma_to_dpx_mappings.tsv")
dme_map <- load_map("rna_dma_to_dme_mappings.tsv")
hsa_map <- load_map("rna_dma_to_hsa_mappings.tsv")      # Ensembl
hsa_gn  <- load_map("rna_dma_to_hsa_gn_mappings.tsv")   # Gene symbols

cat("=== Mapping file coverage (total Dma genes in each file) ===\n")
cat("Dma -> dpx:", nrow(dpx_map), " (", sum(!is.na(dpx_map$ortho)), "with mapping)\n")
cat("Dma -> dme:", nrow(dme_map), " (", sum(!is.na(dme_map$ortho)), "with mapping)\n")
cat("Dma -> hsa:", nrow(hsa_map), " (", sum(!is.na(hsa_map$ortho)), "with mapping)\n")
cat("Dma -> hsa_gn:", nrow(hsa_gn), " (", sum(!is.na(hsa_gn$ortho)), "with mapping)\n\n")

# ---- Step 4: resolve 1:many by picking the FIRST ortholog ------------------
# The mapping files use semicolons to separate multiple orthologs.
# Rule: take the first (typically the primary/best hit). This is a standard
# heuristic when confidence scores aren't available. Document it.

pick_first <- function(x) {
  if (is.na(x)) return(NA_character_)
  trimws(strsplit(x, ";", fixed = TRUE)[[1]][1])
}

dpx_map$ortho_primary <- sapply(dpx_map$ortho, pick_first)
dme_map$ortho_primary <- sapply(dme_map$ortho, pick_first)
hsa_map$ortho_primary <- sapply(hsa_map$ortho, pick_first)
hsa_gn$ortho_primary  <- sapply(hsa_gn$ortho,  pick_first)

# Count how many Dma genes had 1:many collapsed
count_multi <- function(x) sum(!is.na(x) & grepl(";", x))
cat("=== 1:many collapse stats (how many Dma genes had multiple orthologs) ===\n")
cat("dpx: ", count_multi(dpx_map$ortho),
    "of", sum(!is.na(dpx_map$ortho)), "\n")
cat("dme: ", count_multi(dme_map$ortho),
    "of", sum(!is.na(dme_map$ortho)), "\n")
cat("hsa (ensembl):", count_multi(hsa_map$ortho),
    "of", sum(!is.na(hsa_map$ortho)), "\n")
cat("hsa (symbol):",  count_multi(hsa_gn$ortho),
    "of", sum(!is.na(hsa_gn$ortho)), "\n\n")

# ---- Step 1+2: Anchor on dpx ------------------------------------------------
# Only keep Dma genes that have a dpx mapping (the anchor step)
anchored <- dpx_map %>%
  filter(!is.na(ortho_primary)) %>%
  select(dma, dpx_id = ortho_primary)

cat("Step 1 — Dma genes anchored via dpx:", nrow(anchored), "\n")

# ---- Step 3: Cross-reference dme and hsa through the anchor -----------------
# Join against the SAME Dma genes in dme and hsa files
# (i.e., we check whether each anchored gene has a counterpart in other species)
anchored <- anchored %>%
  left_join(dme_map %>% select(dma, dme_flybase = ortho_primary), by = "dma") %>%
  left_join(hsa_map %>% select(dma, hsa_ensembl = ortho_primary), by = "dma") %>%
  left_join(hsa_gn  %>% select(dma, hsa_symbol  = ortho_primary), by = "dma")

# Coverage at each species
cat("\n=== Cross-species coverage (among dpx-anchored Dma genes) ===\n")
cat("With dme ortholog:", sum(!is.na(anchored$dme_flybase)),
    sprintf("(%.1f%%)", 100 * mean(!is.na(anchored$dme_flybase))), "\n")
cat("With hsa Ensembl :", sum(!is.na(anchored$hsa_ensembl)),
    sprintf("(%.1f%%)", 100 * mean(!is.na(anchored$hsa_ensembl))), "\n")
cat("With hsa symbol  :", sum(!is.na(anchored$hsa_symbol)),
    sprintf("(%.1f%%)", 100 * mean(!is.na(anchored$hsa_symbol))), "\n")
cat("\nConfident cross-species mappings (dpx + dme + hsa all present):",
    sum(complete.cases(anchored)), "\n")

# ---- Save the clean mapping table -------------------------------------------
write.csv(anchored,
          "deseq2_results/diagnostics/ortholog_mapping_anchored.csv",
          row.names = FALSE)

# ---- Re-annotate the 29 universal DEGs using the clean mapping --------------
freq <- read.csv("deseq2_results/diagnostics/deg_frequency_across_conditions.csv",
                 stringsAsFactors = FALSE)
core <- freq[freq$n_conditions >= 15, ]

core_anno <- core %>%
  left_join(anchored, by = c("gene_id" = "dma"))

cat("\n=== Universal responders (n=29) annotated via dpx-anchored pipeline ===\n")
cat("With dpx ortholog:", sum(!is.na(core_anno$dpx_id)), "/ 29\n")
cat("With dme ortholog:", sum(!is.na(core_anno$dme_flybase)), "/ 29\n")
cat("With hsa symbol  :", sum(!is.na(core_anno$hsa_symbol)), "/ 29\n")

cat("\nClean annotated table:\n")
print(core_anno, row.names = FALSE)

write.csv(core_anno,
          "deseq2_results/diagnostics/universal_responders_anchored.csv",
          row.names = FALSE)

# ---- Keyword scan on single-gene symbols (much cleaner now) ----------------
cat("\n=== Keyword scan (single primary orthologs, not gene families) ===\n")
syms <- na.omit(core_anno$hsa_symbol)

keyword_check <- function(patterns, label) {
  hits <- character(0)
  for (p in patterns) {
    m <- grep(p, syms, value = TRUE, ignore.case = FALSE)
    if (length(m) > 0) hits <- c(hits, m)
  }
  if (length(hits) > 0) {
    cat(label, ":", paste(unique(hits), collapse = ", "), "\n")
  }
}

keyword_check(c("^CYP", "^GST", "^UGT", "^ABC[BCG]", "^MT[0-9]", "^SULT"),
              "Detox / xenobiotic metabolism")
keyword_check(c("^HSP", "^SOD", "^CAT$", "^GPX", "^FOXO", "NFE2L2"),
              "Stress / oxidative")
keyword_check(c("^FABP", "NPC2", "^LIP", "^ACSL", "^ACSM"),
              "Lipid metabolism")
keyword_check(c("^PRSS", "^KLK", "ELANE", "TRY"),
              "Proteases / digestion")
keyword_check(c("^TLR", "^IL[0-9]", "^NFKB", "^CLEC"),
              "Immune / C-type lectins")

cat("\n=== Done ===\n")
cat("Files written:\n")
cat("  - ortholog_mapping_anchored.csv      (full clean mapping table)\n")
cat("  - universal_responders_anchored.csv  (core 29 with clean annotations)\n")
# ---- ADDITION: also annotate full DEG frequency + top50 with same method ---

# File 1: full DEG frequency table with orthologs
freq_anno <- freq %>%
  left_join(anchored, by = c("gene_id" = "dma")) %>%
  arrange(desc(n_conditions), gene_id)

write.csv(freq_anno,
          "deseq2_results/diagnostics/deg_frequency_with_orthologs.csv",
          row.names = FALSE)
cat("\nWritten: deg_frequency_with_orthologs.csv\n")

# File 2: top 50 pooled-10x DEGs with orthologs
top_pooled_10x <- read.csv("deseq2_results/top50_pooled_10x_DEGs.csv",
                            stringsAsFactors = FALSE)
top_pooled_anno <- top_pooled_10x %>%
  left_join(anchored, by = c("gene_id" = "dma"))

write.csv(top_pooled_anno,
          "deseq2_results/top50_pooled_10x_DEGs_annotated.csv",
          row.names = FALSE)
cat("Written: top50_pooled_10x_DEGs_annotated.csv\n")

## 10. WGCNA Co-expression Network Analysis

In [ ]:
###############################################################################
# DAY 3 — WGCNA on top 5000 variable VST genes
# Approach used by previous cohort (full marks) + aligned with course demo
#
# Pipeline:
#   - Input: rna_vst_counts.csv (VST, not raw)
#   - Filter: top 5000 most variable genes
#   - Exclude low-library samples (D03A6, D03B3) → n=148
#   - Network: unsigned, softPower = 4 (matches course demo)
#   - Method: manual adjacency → TOM → hclust → cutreeDynamic
#   - Correlate eigengenes with metadata (demo) AND 38 chemicals (extension)
###############################################################################

suppressPackageStartupMessages({
  library(WGCNA)
  library(dplyr)
  library(pheatmap)
  library(RColorBrewer)
})

options(stringsAsFactors = FALSE)
dir.create("deseq2_results/wgcna", showWarnings = FALSE, recursive = TRUE)

# ---- Load VST counts + metadata (fileio.R style) ---------------------------
meta_raw <- read.table("sample_sheet.csv", header = TRUE, sep = ",",
                       row.names = 1, check.names = FALSE)
data_raw <- read.table("rna_vst_counts.csv", header = TRUE, sep = ",",
                       row.names = 1, check.names = FALSE)
data_raw <- as.matrix(data_raw)

mids <- match(rownames(meta_raw), colnames(data_raw))
data <- t(data_raw[, mids[!is.na(mids)]])  # samples in rows
meta <- meta_raw[!is.na(mids), ]
stopifnot(rownames(meta) == rownames(data))

cat("Loaded VST:", nrow(data), "samples x", ncol(data), "genes\n")

# ---- Exclude low-library samples -------------------------------------------
exclude <- c("D03A6", "D03B3")
keep <- !rownames(data) %in% exclude
data <- data[keep, ]
meta <- meta[keep, ]
cat("After excluding D03A6, D03B3:", nrow(data), "samples x", ncol(data), "genes\n")

# ---- Filter to top 5000 variable genes -------------------------------------
cat("\n=== Filtering to top 5000 variable genes ===\n")
gene_var <- apply(data, 2, var)
top_n <- 5000
top_genes <- names(sort(gene_var, decreasing = TRUE))[1:top_n]
data <- data[, top_genes]
cat("Filtered:", nrow(data), "samples x", ncol(data), "genes\n")

# ---- Clean check: no NA/NaN/Inf/zero-variance (avoids hclust crash) --------
cat("\n=== Input matrix sanity check ===\n")
cat("Any NA :", any(is.na(data)), "\n")
cat("Any NaN:", any(is.nan(data)), "\n")
cat("Any Inf:", any(is.infinite(data)), "\n")

# Drop any zero-variance genes (belt-and-braces after top-variable filter)
gene_sd <- apply(data, 2, sd)
n_zero_sd <- sum(gene_sd == 0 | is.na(gene_sd))
if (n_zero_sd > 0) {
  cat("Removing", n_zero_sd, "zero-variance genes (defensive)\n")
  data <- data[, !(gene_sd == 0 | is.na(gene_sd))]
}

# WGCNA's own QC
gsg <- goodSamplesGenes(data, verbose = 0)
if (!gsg$allOK) {
  cat("WGCNA QC removing", sum(!gsg$goodGenes), "genes,",
      sum(!gsg$goodSamples), "samples\n")
  data <- data[gsg$goodSamples, gsg$goodGenes]
  meta <- meta[gsg$goodSamples, ]
}
cat("Final matrix:", nrow(data), "samples x", ncol(data), "genes\n")

# ---- Load chemicals for trait correlation ----------------------------------
chem <- read.delim("water_chemicals.tsv", stringsAsFactors = FALSE,
                   check.names = FALSE)

# ---- Step 1: Soft thresholding ---------------------------------------------
cat("\n=== Soft thresholding ===\n")
powers <- 2:20
sft <- pickSoftThreshold(data, powerVector = powers, verbose = 0)

png("deseq2_results/wgcna/soft_threshold.png",
    width = 1400, height = 600, res = 120)
par(mfrow = c(1, 2))
plot(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     xlab = "Soft Threshold (power)",
     ylab = "Scale Free Topology Model Fit, signed R^2",
     type = "n", main = "Scale independence")
text(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     labels = powers, cex = 0.9, col = "red")
abline(h = 0.90, col = "red")

plot(sft$fitIndices[, 1], sft$fitIndices[, 5],
     xlab = "Soft Threshold (power)",
     ylab = "Mean Connectivity",
     type = "n", main = "Mean connectivity")
text(sft$fitIndices[, 1], sft$fitIndices[, 5],
     labels = powers, cex = 0.9, col = "red")
dev.off()

cat("SFT fit table:\n")
print(round(sft$fitIndices[, c(1, 2, 3, 5)], 3))

softPower <- 4
cat("\nUsing softPower =", softPower, "(matches course demo)\n")
cat("[Inspect soft_threshold.png to verify this choice]\n")

# ---- Step 2: Adjacency + TOM (manual) --------------------------------------
cat("\n=== Adjacency matrix (few minutes) ===\n")
adjacency_mat <- adjacency(data, power = softPower)
cat("Done.\n")

cat("\n=== TOM similarity (slowest step) ===\n")
TOM <- TOMsimilarity(adjacency_mat)
dissTOM <- 1 - TOM
rm(adjacency_mat, TOM)  # free memory
gc()
cat("Done.\n")

# ---- Step 3: Clustering + module detection (demo settings) -----------------
cat("\n=== Clustering + module detection ===\n")
geneTree <- hclust(as.dist(dissTOM), method = "average")

dynamicMods <- cutreeDynamic(
  dendro = geneTree,
  distM = dissTOM,
  deepSplit = 2,
  pamRespectsDendro = FALSE,
  minClusterSize = 20
)

moduleColors <- labels2colors(dynamicMods)
cat("\nModule composition (number of genes per module):\n")
print(summary(as.factor(moduleColors)))

module_df <- data.frame(
  gene_id = colnames(data),
  module_num = dynamicMods,
  module_colour = moduleColors
)
write.csv(module_df, "deseq2_results/wgcna/module_gene_assignments.csv",
          row.names = FALSE)

png("deseq2_results/wgcna/module_dendrogram.png",
    width = 1400, height = 700, res = 120)
plotDendroAndColors(geneTree, moduleColors, "Dynamic Tree Cut",
                    dendroLabels = FALSE, hang = 0.03,
                    addGuide = TRUE, guideHang = 0.05,
                    main = "Gene dendrogram and module colors")
dev.off()

# ---- Step 4: Eigengenes + their clustering ---------------------------------
cat("\n=== Computing eigengenes ===\n")
MEList <- moduleEigengenes(data, colors = moduleColors)
MEs <- MEList$eigengenes

MEDiss <- 1 - cor(MEs)
METree <- hclust(as.dist(MEDiss), method = "average")

png("deseq2_results/wgcna/eigengene_clustering.png",
    width = 900, height = 600, res = 120)
plot(METree, main = "Clustering of module eigengenes",
     xlab = "", sub = "")
dev.off()

# ---- Step 5: Module-metadata correlations (demo style) ---------------------
cat("\n=== Module-metadata correlations (matches demo) ===\n")
conds <- lapply(meta, function(x) as.numeric(as.factor(x)))
conds <- as.data.frame(conds, row.names = rownames(meta))

moduleTraitCor    <- cor(MEs, conds, use = "p")
moduleTraitPvalue <- corPvalueStudent(moduleTraitCor, nrow(data))

png("deseq2_results/wgcna/module_trait_metadata.png",
    width = 1100, height = 1400, res = 120)
textMatrix <- paste(signif(moduleTraitCor, 2), "\n(",
                    signif(moduleTraitPvalue, 1), ")", sep = "")
dim(textMatrix) <- dim(moduleTraitCor)
par(mar = c(6, 8.5, 3, 3))
labeledHeatmap(
  Matrix = moduleTraitCor,
  xLabels = names(conds),
  yLabels = names(MEs),
  ySymbols = names(MEs),
  colorLabels = FALSE,
  colors = blueWhiteRed(50),
  textMatrix = textMatrix,
  setStdMargins = FALSE,
  cex.text = 0.6,
  zlim = c(-1, 1),
  main = "Module-trait relationships (metadata)"
)
dev.off()

# ---- Step 6: EXTENSION — module-chemical correlations ----------------------
cat("\n=== Module-chemical correlations (extension) ===\n")

rownames(chem) <- chem[[1]]
site_cols_chem <- grep("^D", colnames(chem), value = TRUE)
chem_mat <- as.matrix(chem[, site_cols_chem])

chem_per_sample <- matrix(0, nrow = nrow(meta), ncol = nrow(chem),
                          dimnames = list(rownames(meta), rownames(chem)))
for (i in seq_len(nrow(meta))) {
  ref <- meta$REF[i]
  site_key <- meta$Site[i]
  if (ref == "Control") next
  if (!site_key %in% colnames(chem_mat)) next
  multiplier <- ifelse(ref == "10x", 10, 1)
  chem_per_sample[i, ] <- chem_mat[, site_key] * multiplier
}

chem_kept <- chem_per_sample[, colSums(chem_per_sample) > 0]
cat("Chemicals detected at >=1 site:", ncol(chem_kept), "\n")

# Correlate on treated samples only (controls are all zeros for chemicals)
treated_idx <- meta$REF != "Control"
MEs_treated  <- MEs[treated_idx, , drop = FALSE]
chem_treated <- chem_kept[treated_idx, , drop = FALSE]

moduleChemCor    <- cor(MEs_treated, chem_treated, use = "p")
moduleChemPvalue <- corPvalueStudent(moduleChemCor, nrow(MEs_treated))

write.csv(moduleChemCor,
          "deseq2_results/wgcna/module_chemical_correlations.csv")
write.csv(moduleChemPvalue,
          "deseq2_results/wgcna/module_chemical_pvalues.csv")

breaks  <- seq(-1, 1, length.out = 101)
colours <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(100)

png("deseq2_results/wgcna/module_chemical_heatmap.png",
    width = 2200, height = 1100, res = 120)
pheatmap(moduleChemCor,
         color = colours, breaks = breaks,
         cluster_rows = TRUE, cluster_cols = TRUE,
         show_colnames = TRUE, show_rownames = TRUE,
         fontsize_col = 6, fontsize_row = 9,
         main = sprintf("Module-chemical correlations (%d modules x %d chemicals)",
                        nrow(moduleChemCor), ncol(moduleChemCor)))
dev.off()

# ---- Top modules by chemical association -----------------------------------
cat("\n=== Top modules by strongest chemical association ===\n")
abs_max <- apply(abs(moduleChemCor), 1, max)
top_modules <- sort(abs_max, decreasing = TRUE)[1:min(10, length(abs_max))]
cat("Top 10 modules by max |r|:\n")
print(round(top_modules, 3))

cat("\nTop 3 chemicals per top module:\n")
for (mod in names(top_modules)) {
  mod_cors <- moduleChemCor[mod, ]
  top_chems <- sort(abs(mod_cors), decreasing = TRUE)[1:3]
  mod_name_no_ME <- sub("^ME", "", mod)
  n_genes <- sum(moduleColors == mod_name_no_ME)
  cat(sprintf("\n  %s (%d genes):\n", mod, n_genes))
  for (chem_nm in names(top_chems)) {
    r <- mod_cors[chem_nm]
    p <- moduleChemPvalue[mod, chem_nm]
    cat(sprintf("    %-50s  r=%+.3f  p=%.2e\n",
                substr(chem_nm, 1, 50), r, p))
  }
}

# ---- Save outputs ----------------------------------------------------------
write.csv(MEs, "deseq2_results/wgcna/module_eigengenes.csv")
write.csv(moduleTraitCor,
          "deseq2_results/wgcna/module_trait_metadata_cor.csv")
write.csv(moduleTraitPvalue,
          "deseq2_results/wgcna/module_trait_metadata_pval.csv")

# ---- Save the top-5000-variable VST subset for iDEP upload -----------------
# This is a convenience output: a CSV you can directly upload to iDEP
idep_out <- t(data)  # genes in rows for iDEP
write.csv(idep_out, "deseq2_results/wgcna/vst_top5000_for_iDEP.csv")
cat("\nSaved: vst_top5000_for_iDEP.csv — ready to upload to iDEP web tool\n")

cat("\n=== WGCNA done ===\n")
cat("Network built on", ncol(data), "top-variable genes across",
    nrow(data), "samples\n")
cat("Outputs in deseq2_results/wgcna/\n")

In [ ]:
###############################################################################
# DAY 3.5 — Deep-dive into MEblue (Diuron-linked) module
#
# Part 1: Extract the 741 MEblue genes, annotate with orthologs, save for
#         pathway teammate
# Part 2: Map Diuron concentration across sites; cross-reference with
#         per-site DEG counts to see whether Diuron hotspots match our
#         high-DEG sites (D11, D12, D06, etc.)
# Part 3: Visualise MEblue eigengene values across sites / concentrations
###############################################################################

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pheatmap)
  library(RColorBrewer)
})

dir.create("deseq2_results/wgcna/meblue_deepdive",
           showWarnings = FALSE, recursive = TRUE)

# ============================================================================
# PART 1: Extract + annotate MEblue genes
# ============================================================================
cat("=== PART 1: MEblue gene extraction + annotation ===\n")

modules <- read.csv("deseq2_results/wgcna/module_gene_assignments.csv",
                    stringsAsFactors = FALSE)
meblue_genes <- modules$gene_id[modules$module_colour == "blue"]
cat("MEblue module:", length(meblue_genes), "genes\n")

# Load the clean dpx-anchored ortholog table from the earlier step
ortho <- read.csv("deseq2_results/diagnostics/ortholog_mapping_anchored.csv",
                  stringsAsFactors = FALSE)

meblue_anno <- data.frame(gene_id = meblue_genes, stringsAsFactors = FALSE)
meblue_anno <- meblue_anno %>%
  left_join(ortho, by = c("gene_id" = "dma"))

# Overlap with DEG frequency
deg_freq <- read.csv("deseq2_results/diagnostics/deg_frequency_across_conditions.csv",
                     stringsAsFactors = FALSE)
meblue_anno <- meblue_anno %>%
  left_join(deg_freq, by = "gene_id")
meblue_anno$n_conditions[is.na(meblue_anno$n_conditions)] <- 0

cat("MEblue genes with dpx ortholog :", sum(!is.na(meblue_anno$dpx_id)), "\n")
cat("MEblue genes with human symbol :", sum(!is.na(meblue_anno$hsa_symbol)), "\n")
cat("MEblue genes that are also DEG :", sum(meblue_anno$n_conditions > 0), "\n")
cat("MEblue genes in universal (>=15 conditions):",
    sum(meblue_anno$n_conditions >= 15), "\n")

# Sort: DEGs first, then by generality
meblue_anno <- meblue_anno %>%
  arrange(desc(n_conditions), gene_id)

write.csv(meblue_anno,
          "deseq2_results/wgcna/meblue_deepdive/meblue_genes_annotated.csv",
          row.names = FALSE)

# Keyword scan on the human symbols
cat("\n=== Functional keyword scan on MEblue human symbols ===\n")
syms <- na.omit(meblue_anno$hsa_symbol)

keyword_check <- function(patterns, label) {
  hits <- character(0)
  for (p in patterns) {
    m <- grep(p, syms, value = TRUE, ignore.case = FALSE)
    if (length(m) > 0) hits <- c(hits, m)
  }
  hits <- unique(hits)
  if (length(hits) > 0) {
    cat(sprintf("%-35s : %d hits\n", label, length(hits)))
    cat("  ", paste(hits, collapse = ", "), "\n")
  } else {
    cat(sprintf("%-35s : none\n", label))
  }
}

keyword_check(c("^CYP"),           "Cytochrome P450 (phase I)")
keyword_check(c("^GST"),           "Glutathione S-transferase")
keyword_check(c("^UGT"),           "UDP-glucuronosyltransferase")
keyword_check(c("^SULT"),          "Sulfotransferase")
keyword_check(c("^ABC[BCG]"),      "ABC transporter (drug efflux)")
keyword_check(c("^MT[0-9]", "MT1", "MT2"),  "Metallothionein")
keyword_check(c("^HSP", "^HSPA", "^HSPB"),  "Heat shock protein")
keyword_check(c("^SOD", "^CAT$", "^GPX"),    "Oxidative stress")
keyword_check(c("^FABP", "NPC2", "^LIP", "^ACSL", "^ACSM"),
              "Lipid metabolism")
keyword_check(c("^PRSS", "^KLK", "ELANE", "^TRY"),
              "Proteases / digestion")
keyword_check(c("^TLR", "^CLEC", "^IL[0-9]"),
              "Immune")
keyword_check(c("^RPL", "^RPS"), "Ribosomal (housekeeping)")

# ============================================================================
# PART 2: Diuron distribution across sites + DEG cross-reference
# ============================================================================
cat("\n\n=== PART 2: Diuron spatial distribution ===\n")

chem <- read.delim("water_chemicals.tsv", stringsAsFactors = FALSE,
                   check.names = FALSE)

# Find the Diuron row
rownames(chem) <- chem[[1]]
diuron_row <- grep("Diuron", rownames(chem), value = TRUE, ignore.case = TRUE)
cat("Diuron entries found:", diuron_row, "\n")

# Extract per-site concentration
site_cols_chem <- grep("^D", colnames(chem), value = TRUE)
diuron_values <- as.numeric(chem[diuron_row[1], site_cols_chem])
names(diuron_values) <- site_cols_chem

cat("\nDiuron concentration per site (environmental 1x, ug/L):\n")
for (s in names(diuron_values)) {
  cat(sprintf("  %s : %.4f\n", s, diuron_values[s]))
}

diuron_df <- data.frame(
  site = factor(names(diuron_values), levels = names(diuron_values)),
  diuron_ug_L = diuron_values,
  stringsAsFactors = FALSE
)

# Cross-reference with per-site DEG counts
deg_per_site_10x <- read.csv("deseq2_results/per_site_10x_summary.csv",
                              stringsAsFactors = FALSE)
deg_per_site_1x  <- read.csv("deseq2_results/per_site_1x_summary.csv",
                              stringsAsFactors = FALSE)

xref <- diuron_df %>%
  left_join(deg_per_site_10x %>% select(site, n_deg_10x = n_deg), by = "site") %>%
  left_join(deg_per_site_1x  %>% select(site, n_deg_1x  = n_deg), by = "site")

cat("\n=== Diuron vs DEG counts per site ===\n")
print(xref)

write.csv(xref,
          "deseq2_results/wgcna/meblue_deepdive/diuron_vs_degs_per_site.csv",
          row.names = FALSE)

# Correlation test
cor_10x <- cor.test(xref$diuron_ug_L, xref$n_deg_10x)
cor_1x  <- cor.test(xref$diuron_ug_L, xref$n_deg_1x)
cat(sprintf("\nCorrelation Diuron vs n_DEGs (10x): r=%.3f, p=%.3f\n",
            cor_10x$estimate, cor_10x$p.value))
cat(sprintf("Correlation Diuron vs n_DEGs (1x) : r=%.3f, p=%.3f\n",
            cor_1x$estimate,  cor_1x$p.value))

# Plot: Diuron per site + DEG counts overlay
p_diuron <- ggplot(diuron_df, aes(site, diuron_ug_L)) +
  geom_col(fill = "#e41a1c") +
  labs(title = "Diuron concentration per site (environmental level)",
       subtitle = "Herbicide concentration along Danube (upstream D01 -> downstream D12)",
       x = "Site", y = "Diuron (ug/L)") +
  theme_minimal(base_size = 13)

ggsave("deseq2_results/wgcna/meblue_deepdive/diuron_per_site.png",
       p_diuron, width = 9, height = 5, dpi = 150)

# Scatter: Diuron vs DEGs
xref_long <- xref %>%
  pivot_longer(cols = starts_with("n_deg"),
               names_to = "dose", values_to = "n_deg") %>%
  mutate(dose = ifelse(dose == "n_deg_10x", "10x", "1x"))

p_scatter <- ggplot(xref_long, aes(diuron_ug_L, n_deg, colour = dose)) +
  geom_point(size = 3, alpha = 0.9) +
  geom_text(aes(label = site), vjust = -0.8, size = 3.5, show.legend = FALSE) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 0.5, linetype = "dashed") +
  scale_colour_manual(values = c("1x" = "#377eb8", "10x" = "#e41a1c")) +
  labs(title = "Site-level Diuron vs transcriptional response",
       subtitle = "Does Diuron concentration predict number of DEGs per site?",
       x = "Diuron concentration (ug/L)",
       y = "Number of DEGs per site") +
  theme_minimal(base_size = 13)

ggsave("deseq2_results/wgcna/meblue_deepdive/diuron_vs_DEGs_scatter.png",
       p_scatter, width = 9, height = 6, dpi = 150)

# ============================================================================
# PART 3: MEblue eigengene across sites + concentrations
# ============================================================================
cat("\n\n=== PART 3: MEblue eigengene expression pattern ===\n")

MEs <- read.csv("deseq2_results/wgcna/module_eigengenes.csv",
                stringsAsFactors = FALSE, row.names = 1)

ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)
ss <- ss[!ss$SampleID %in% c("D03A6", "D03B3"), ]

# Match eigengene rows to sample sheet
meblue_df <- data.frame(
  sample_id = rownames(MEs),
  MEblue = MEs$MEblue
)
meblue_df <- meblue_df %>%
  left_join(ss %>% select(sample_id = SampleID, site = Site, conc = REF),
            by = "sample_id")

meblue_df$conc <- factor(meblue_df$conc, levels = c("Control", "1x", "10x"))

p_eigengene <- ggplot(meblue_df, aes(site, MEblue, fill = conc)) +
  geom_boxplot(alpha = 0.85, outlier.size = 1) +
  scale_fill_manual(values = c("Control" = "#888888",
                               "1x" = "#66a5e0",
                               "10x" = "#e41a1c")) +
  labs(title = "MEblue eigengene across sites and concentrations",
       subtitle = "Higher values = gene module more active (positively correlated with Diuron, r=0.585)",
       x = NULL, y = "MEblue eigengene") +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "top")

ggsave("deseq2_results/wgcna/meblue_deepdive/meblue_eigengene_boxplot.png",
       p_eigengene, width = 10, height = 6, dpi = 150)

# Summary: mean MEblue per site x concentration
meblue_summary <- meblue_df %>%
  group_by(site, conc) %>%
  summarise(mean_MEblue = mean(MEblue), .groups = "drop")

write.csv(meblue_summary,
          "deseq2_results/wgcna/meblue_deepdive/meblue_mean_per_site_dose.csv",
          row.names = FALSE)

cat("\nMEblue eigengene — mean per site x dose:\n")
print(meblue_summary %>%
        pivot_wider(names_from = conc, values_from = mean_MEblue) %>%
        as.data.frame(), row.names = FALSE)

cat("\n=== DONE ===\n")
cat("Outputs in deseq2_results/wgcna/meblue_deepdive/:\n")
cat("  - meblue_genes_annotated.csv        (741 genes with orthologs + DEG status)\n")
cat("  - diuron_per_site.png                (Diuron bar chart)\n")
cat("  - diuron_vs_DEGs_scatter.png         (Diuron-DEG relationship)\n")
cat("  - diuron_vs_degs_per_site.csv        (underlying table)\n")
cat("  - meblue_eigengene_boxplot.png       (MEblue expression pattern) *** KEY FIGURE ***\n")
cat("  - meblue_mean_per_site_dose.csv\n")

In [ ]:
###############################################################################
# DAY 3.6 — Find chemicals driving D11/D12 toxicity
#
# D11 and D12 have the most DEGs (~1,600 each at 1x) but zero Diuron.
# Question: which chemicals ARE elevated at D11/D12, and which WGCNA modules
# correlate with those chemicals?
###############################################################################

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pheatmap)
  library(RColorBrewer)
})

dir.create("deseq2_results/wgcna/d11_d12_chemistry",
           showWarnings = FALSE, recursive = TRUE)

# ---- Load chemistry table --------------------------------------------------
chem <- read.delim("water_chemicals.tsv", stringsAsFactors = FALSE,
                   check.names = FALSE)
rownames(chem) <- chem[[1]]
site_cols <- grep("^D", colnames(chem), value = TRUE)
chem_mat <- as.matrix(chem[, site_cols])

cat("Chemistry table:", nrow(chem_mat), "chemicals x", ncol(chem_mat), "sites\n")

# ---- Identify chemicals enriched at D11/D12 --------------------------------
# Strategy: rank each chemical by its (D11+D12) sum vs the rest-of-sites sum
# Higher ratio = more downstream-specific

downstream <- c("D11", "D12")
upstream   <- setdiff(site_cols, downstream)

cat("\n=== Chemicals elevated at D11/D12 ===\n")

chem_summary <- data.frame(
  chemical = rownames(chem_mat),
  d11 = chem_mat[, "D11"],
  d12 = chem_mat[, "D12"],
  downstream_mean = rowMeans(chem_mat[, downstream, drop = FALSE]),
  rest_mean = rowMeans(chem_mat[, upstream, drop = FALSE]),
  stringsAsFactors = FALSE
)

# Compute enrichment: downstream / (rest + small constant to avoid div-by-zero)
chem_summary$enrichment_ratio <- with(chem_summary,
                                       (downstream_mean + 1e-6) /
                                       (rest_mean + 1e-6))

# Filter: keep only chemicals detected at D11 or D12
chem_summary <- chem_summary[chem_summary$d11 > 0 | chem_summary$d12 > 0, ]
cat("Chemicals detected at D11 or D12:", nrow(chem_summary), "\n")

# Top by absolute concentration at D11/D12
top_by_conc <- chem_summary %>%
  arrange(desc(downstream_mean)) %>%
  head(15)

cat("\nTop 15 chemicals by ABSOLUTE concentration at D11/D12 (ug/L):\n")
print(top_by_conc[, c("chemical", "d11", "d12", "rest_mean")],
      row.names = FALSE)

# Top by enrichment (relative to rest of river)
top_by_enrichment <- chem_summary %>%
  filter(downstream_mean > 0.01) %>%   # filter out trace amounts
  arrange(desc(enrichment_ratio)) %>%
  head(15)

cat("\nTop 15 chemicals MOST ENRICHED at D11/D12 vs other sites:\n")
print(top_by_enrichment[, c("chemical", "d11", "d12", "rest_mean", "enrichment_ratio")],
      row.names = FALSE)

# Chemicals UNIQUE to D11/D12 (zero elsewhere)
unique_to_dwn <- chem_summary %>%
  filter(rest_mean == 0, downstream_mean > 0) %>%
  arrange(desc(downstream_mean))

cat(sprintf("\nChemicals found ONLY at D11/D12 (zero at other sites): %d\n",
            nrow(unique_to_dwn)))
if (nrow(unique_to_dwn) > 0) {
  print(unique_to_dwn[, c("chemical", "d11", "d12")], row.names = FALSE)
}

write.csv(chem_summary,
          "deseq2_results/wgcna/d11_d12_chemistry/all_chemicals_d11_d12_summary.csv",
          row.names = FALSE)

# ---- Plot: top D11/D12 chemicals across all sites --------------------------
top_chems_to_plot <- head(top_by_conc$chemical, 12)

chem_long <- chem_mat[top_chems_to_plot, ] %>%
  as.data.frame() %>%
  tibble::rownames_to_column("chemical") %>%
  pivot_longer(-chemical, names_to = "site", values_to = "concentration")

chem_long$site <- factor(chem_long$site, levels = site_cols)
chem_long$chemical <- factor(chem_long$chemical, levels = top_chems_to_plot)

p_top_chems <- ggplot(chem_long, aes(site, concentration, fill = site)) +
  geom_col() +
  facet_wrap(~ chemical, scales = "free_y", ncol = 3) +
  scale_fill_manual(values = c(rep("#666666", 10), "#e41a1c", "#e41a1c")) +
  labs(title = "Top chemicals at D11/D12 vs upstream sites",
       subtitle = "Red = downstream (D11/D12), grey = others",
       x = NULL, y = "Concentration (ug/L)") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
        legend.position = "none")

ggsave("deseq2_results/wgcna/d11_d12_chemistry/top_chems_per_site.png",
       p_top_chems, width = 11, height = 9, dpi = 150)

# ---- Cross-reference with WGCNA module-chemical correlations ---------------
cat("\n=== WGCNA modules linked to D11/D12 chemicals ===\n")

mod_chem_cor <- read.csv("deseq2_results/wgcna/module_chemical_correlations.csv",
                         stringsAsFactors = FALSE, row.names = 1, check.names = FALSE)
mod_chem_p   <- read.csv("deseq2_results/wgcna/module_chemical_pvalues.csv",
                         stringsAsFactors = FALSE, row.names = 1, check.names = FALSE)

# For each top D11/D12 chemical, which modules correlate most strongly?
top_chems_in_wgcna <- intersect(top_chems_to_plot, colnames(mod_chem_cor))

cat("D11/D12 top chemicals also analysed by WGCNA:",
    length(top_chems_in_wgcna), "/", length(top_chems_to_plot), "\n\n")

for (ch in top_chems_in_wgcna) {
  cors <- mod_chem_cor[, ch]
  pvals <- mod_chem_p[, ch]
  names(cors) <- rownames(mod_chem_cor)
  top_mods <- sort(abs(cors), decreasing = TRUE)[1:3]
  cat(sprintf("\n  %s (D11=%.3f, D12=%.3f ug/L):\n",
              ch,
              chem_summary$d11[chem_summary$chemical == ch],
              chem_summary$d12[chem_summary$chemical == ch]))
  for (mod in names(top_mods)) {
    r <- cors[mod]
    p <- pvals[which(rownames(mod_chem_cor) == mod), ch]
    cat(sprintf("    %-15s  r=%+.3f  p=%.2e\n", mod, r, p))
  }
}

# ---- Subset module-chemical heatmap to just D11/D12 chemicals --------------
if (length(top_chems_in_wgcna) > 1) {
  sub_cor <- mod_chem_cor[, top_chems_in_wgcna, drop = FALSE]
  sub_cor <- as.matrix(sub_cor)

  breaks  <- seq(-1, 1, length.out = 101)
  colours <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(100)

  png("deseq2_results/wgcna/d11_d12_chemistry/module_d11d12_chemicals_heatmap.png",
      width = 1600, height = 1100, res = 120)
  pheatmap(sub_cor,
           color = colours, breaks = breaks,
           cluster_rows = TRUE, cluster_cols = TRUE,
           show_colnames = TRUE, show_rownames = TRUE,
           fontsize_col = 10, fontsize_row = 10,
           main = sprintf("WGCNA modules vs top %d D11/D12 chemicals",
                          length(top_chems_in_wgcna)))
  dev.off()
}

cat("\n=== DONE ===\n")
cat("Outputs in deseq2_results/wgcna/d11_d12_chemistry/:\n")
cat("  - all_chemicals_d11_d12_summary.csv      (full chemistry breakdown)\n")
cat("  - top_chems_per_site.png                 (top 12 D11/D12 chemicals across river)\n")
cat("  - module_d11d12_chemicals_heatmap.png    (which modules respond to these)\n")

In [ ]:
###############################################################################
# Fix for the indexing error + complete the D11/D12 chemistry analysis
# Run this AFTER d11_d12_chemistry.R has loaded its variables
# (or just paste this into the same R session)
###############################################################################

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pheatmap)
  library(RColorBrewer)
})

# Reload if needed
chem <- read.delim("water_chemicals.tsv", stringsAsFactors = FALSE,
                   check.names = FALSE)
rownames(chem) <- chem[[1]]
site_cols <- grep("^D", colnames(chem), value = TRUE)
chem_mat <- as.matrix(chem[, site_cols])

downstream <- c("D11", "D12")
upstream <- setdiff(site_cols, downstream)

chem_summary <- data.frame(
  chemical = rownames(chem_mat),
  d11 = chem_mat[, "D11"],
  d12 = chem_mat[, "D12"],
  downstream_mean = rowMeans(chem_mat[, downstream, drop = FALSE]),
  rest_mean = rowMeans(chem_mat[, upstream, drop = FALSE]),
  stringsAsFactors = FALSE
)
chem_summary$enrichment_ratio <- with(chem_summary,
                                       (downstream_mean + 1e-6) / (rest_mean + 1e-6))
chem_summary <- chem_summary[chem_summary$d11 > 0 | chem_summary$d12 > 0, ]

top_by_conc <- chem_summary %>% arrange(desc(downstream_mean)) %>% head(15)

cat("\n=== FULL: Top 15 by absolute concentration at D11/D12 ===\n")
print(top_by_conc[, c("chemical", "d11", "d12", "rest_mean")], row.names = FALSE)

top_by_enrichment <- chem_summary %>%
  filter(downstream_mean > 0.01) %>%
  arrange(desc(enrichment_ratio)) %>%
  head(15)

cat("\n=== FULL: Top 15 by enrichment vs rest of river ===\n")
print(top_by_enrichment[, c("chemical", "d11", "d12", "rest_mean", "enrichment_ratio")],
      row.names = FALSE)

unique_to_dwn <- chem_summary %>%
  filter(rest_mean == 0, downstream_mean > 0) %>%
  arrange(desc(downstream_mean))
cat(sprintf("\n=== %d chemicals UNIQUE to D11/D12 ===\n", nrow(unique_to_dwn)))
if (nrow(unique_to_dwn) > 0) {
  print(unique_to_dwn[, c("chemical", "d11", "d12")], row.names = FALSE)
}

# ---- FIXED cross-reference with WGCNA modules ------------------------------
mod_chem_cor <- read.csv("deseq2_results/wgcna/module_chemical_correlations.csv",
                         stringsAsFactors = FALSE, row.names = 1, check.names = FALSE)
mod_chem_p   <- read.csv("deseq2_results/wgcna/module_chemical_pvalues.csv",
                         stringsAsFactors = FALSE, row.names = 1, check.names = FALSE)

top_chems_to_plot <- head(top_by_conc$chemical, 12)
top_chems_in_wgcna <- intersect(top_chems_to_plot, colnames(mod_chem_cor))

cat(sprintf("\n=== WGCNA modules linked to D11/D12 top chemicals (%d analysed) ===\n",
            length(top_chems_in_wgcna)))

for (ch in top_chems_in_wgcna) {
  cors  <- mod_chem_cor[, ch]    # vector indexed by module name
  pvals <- mod_chem_p[, ch]      # vector indexed by module name
  names(cors)  <- rownames(mod_chem_cor)
  names(pvals) <- rownames(mod_chem_p)

  top_mods <- sort(abs(cors), decreasing = TRUE)[1:3]
  cat(sprintf("\n  %s (D11=%.3f, D12=%.3f ug/L):\n",
              ch,
              chem_summary$d11[chem_summary$chemical == ch],
              chem_summary$d12[chem_summary$chemical == ch]))
  for (mod in names(top_mods)) {
    r <- cors[mod]
    p <- pvals[mod]            # FIXED: just one index
    cat(sprintf("    %-18s  r=%+.3f  p=%.2e\n", mod, r, p))
  }
}

# ---- Subset heatmap of these chemicals -------------------------------------
if (length(top_chems_in_wgcna) > 1) {
  sub_cor <- as.matrix(mod_chem_cor[, top_chems_in_wgcna, drop = FALSE])
  breaks  <- seq(-1, 1, length.out = 101)
  colours <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(100)

  png("deseq2_results/wgcna/d11_d12_chemistry/module_d11d12_chemicals_heatmap.png",
      width = 1600, height = 1100, res = 120)
  pheatmap(sub_cor,
           color = colours, breaks = breaks,
           cluster_rows = TRUE, cluster_cols = TRUE,
           show_colnames = TRUE, show_rownames = TRUE,
           fontsize_col = 10, fontsize_row = 10,
           main = sprintf("WGCNA modules vs top %d D11/D12 chemicals",
                          length(top_chems_in_wgcna)))
  dev.off()
  cat("\nSaved heatmap.\n")
}

# ---- Plot top D11/D12 chemicals across all sites ---------------------------
chem_long <- chem_mat[top_chems_to_plot, , drop = FALSE] %>%
  as.data.frame() %>%
  tibble::rownames_to_column("chemical") %>%
  pivot_longer(-chemical, names_to = "site", values_to = "concentration")

chem_long$site <- factor(chem_long$site, levels = site_cols)
chem_long$chemical <- factor(chem_long$chemical, levels = top_chems_to_plot)

p_top_chems <- ggplot(chem_long, aes(site, concentration, fill = site)) +
  geom_col() +
  facet_wrap(~ chemical, scales = "free_y", ncol = 3) +
  scale_fill_manual(values = c(rep("#666666", 10), "#e41a1c", "#e41a1c")) +
  labs(title = "Top chemicals at D11/D12 vs upstream sites",
       subtitle = "Red bars = downstream (D11/D12), grey = others",
       x = NULL, y = "Concentration (ug/L)") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
        legend.position = "none")

ggsave("deseq2_results/wgcna/d11_d12_chemistry/top_chems_per_site.png",
       p_top_chems, width = 11, height = 9, dpi = 150)
cat("Saved top_chems_per_site.png\n")

write.csv(chem_summary,
          "deseq2_results/wgcna/d11_d12_chemistry/all_chemicals_d11_d12_summary.csv",
          row.names = FALSE)

cat("\n=== DONE ===\n")

## 11. Package Outputs for Team

In [ ]:
###############################################################################
# Package transcriptomics outputs for sharing with teammates — R version
# Run this from the same R session you've been using
###############################################################################

OUTDIR <- "share_with_team"
dir.create(OUTDIR, showWarnings = FALSE)

# Helper to zip a list of files into one archive
make_zip <- function(zipfile, files, readme_text) {
  files_existing <- files[file.exists(files)]
  files_missing  <- files[!file.exists(files)]

  if (length(files_missing) > 0) {
    cat("[!] Skipped (not found):", paste(files_missing, collapse = ", "), "\n")
  }

  # Write the README to a temp file
  readme_path <- file.path(tempdir(), "README.txt")
  writeLines(readme_text, readme_path)

  files_to_zip <- c(files_existing, readme_path)

  # zip(), with -j to flatten paths so files end up at the zip root
  zip::zip(zipfile = file.path(OUTDIR, zipfile),
           files = files_to_zip,
           mode = "cherry-pick")  # keeps just the basenames

  cat("[OK]", zipfile, "(",
      length(files_existing) + 1, "files )\n")
}

# Make sure the zip package is installed (more reliable than base::zip on Windows)

# ============================================================================
# 1. Pathway enrichment package
# ============================================================================
cat("\n=== Building pathway enrichment package ===\n")

readme_pathway <- "PATHWAY ENRICHMENT INPUTS
=========================

Use these files for GO/KEGG/Reactome enrichment analysis.

KEY FILES:
- A_pooled_10x_vs_Control.csv : full ranked DEG list, all 17,881 genes
                                (use for GSEA-style methods needing ranked lists)
- B_pooled_1x_vs_Control.csv  : same for low-dose contrast
- universal_responders_anchored.csv : 29 universal pollution responders
                                       (DEG in 15+ of 24 conditions)
- top50_pooled_10x_DEGs_annotated.csv : top 50 DEGs with human ortholog mapped
- meblue_genes_annotated.csv : 741 genes in WGCNA MEblue module
                                (Diuron-linked detox cluster)

ORTHOLOG MAPPING:
- ortholog_mapping_anchored.csv : full Dma -> dpx -> dme -> hsa table
  Anchored on D. pulex (closest species), then propagated.
  Resolves 1:many by primary (first-listed) ortholog.

DEG THRESHOLD USED: padj < 0.05 AND |log2FC| > 1

QUESTIONS TO DECIDE:
1. Filtered DEG list (cut at threshold) vs. full ranked list?
   - g:Profiler / DAVID / WebGestalt ORA: cut list works fine
   - GSEA / fgsea: prefer the FULL ranked list (use A_pooled_10x_vs_Control.csv)
2. Which species ortholog for enrichment?
   - Human (hsa_symbol or hsa_ensembl) gives best DB coverage
   - Drosophila (dme_flybase) is a closer evolutionary match"

make_zip(
  zipfile = "for_pathway_enrichment.zip",
  files = c(
    "deseq2_results/A_pooled_10x_vs_Control.csv",
    "deseq2_results/B_pooled_1x_vs_Control.csv",
    "deseq2_results/top50_pooled_10x_DEGs.csv",
    "deseq2_results/top50_pooled_10x_DEGs_annotated.csv",
    "deseq2_results/diagnostics/universal_responders_anchored.csv",
    "deseq2_results/diagnostics/deg_frequency_with_orthologs.csv",
    "deseq2_results/diagnostics/ortholog_mapping_anchored.csv",
    "deseq2_results/wgcna/meblue_deepdive/meblue_genes_annotated.csv"
  ),
  readme_text = readme_pathway
)

# ============================================================================
# 2. Multi-omics integration package
# ============================================================================
cat("\n=== Building integration package ===\n")

readme_integration <- "MULTI-OMICS INTEGRATION INPUTS
==============================

For integrating transcriptomics with metabolomics (e.g. via MOFA, RGCCA, SGCCA).

PRIMARY INTEGRATION FEATURES:
- module_eigengenes.csv : 17 WGCNA module eigengenes per sample (148 samples)
  RECOMMENDED transcriptomic input for MOFA — reduces dimensionality
  from 5000 genes to ~17 modules, each with biological meaning.

GENE-LEVEL DATA (if needed):
- vst_top5000_for_iDEP.csv : top 5000 variable VST genes x 148 samples
  Use only if your method requires gene-level features.

CONTEXT FOR INTERPRETATION:
- module_gene_assignments.csv : which genes are in each module
- module_chemical_correlations.csv : how each module correlates with the 38
  chemicals — helps you link transcriptomic factors to chemistry
- sample_sheet.csv : metadata (148 rows: SampleID, REF, Site, Description)

SAMPLE EXCLUSIONS:
2 samples removed for low library size: D03A6, D03B3
Final n = 148 samples

KEY MODULES BY CHEMICAL ASSOCIATION:
- MEblue (741 genes)  : strongest correlation with Diuron (r=+0.585)
                        Also responds to wastewater chemicals (Acesulfame, Sucralose,
                        Benzotriazoles). Enriched for detox/lipid metabolism.
- MEpink, MEmagenta   : co-respond with MEblue, similar chemistry
- MEpurple            : negatively correlates with pollution (suppressed biology)

MOFA RECOMMENDATION (from previous cohort, full marks):
'Other tools have limitations for transcriptomics-metabolomics integration —
 MOFA gives factors common between omics layers, strongest approach.'"

make_zip(
  zipfile = "for_integration.zip",
  files = c(
    "deseq2_results/wgcna/module_eigengenes.csv",
    "deseq2_results/wgcna/module_gene_assignments.csv",
    "deseq2_results/wgcna/module_chemical_correlations.csv",
    "deseq2_results/wgcna/vst_top5000_for_iDEP.csv",
    "sample_sheet.csv"
  ),
  readme_text = readme_integration
)

# ============================================================================
# 3. Shared summary package
# ============================================================================
cat("\n=== Building shared summary package ===\n")

readme_shared <- "SHARED TRANSCRIPTOMICS RESULTS
==============================

Summary plots and tables for everyone.

SITE RANKINGS:
- per_site_10x_summary.csv / per_site_1x_summary.csv
- degs_per_site.png : bar plot

KEY FINDINGS:
- D11 (1570 DEGs) and D12 (1605 DEGs) at 1x are the most affected sites
- D02, D03, D04, D09 dominate at 10x
- D08 at 10x: 0 DEGs (anomaly)

WGCNA MODULES:
- module_dendrogram.png : 17 modules from 5000 most variable genes
- module_chemical_heatmap.png : KEY FIGURE — module-chemical correlations
- module_trait_metadata.png : module vs site/REF
- meblue_eigengene_boxplot.png : MEblue (Diuron-linked) across sites

CHEMISTRY AT D11/D12:
- top_chems_per_site.png : wastewater chemicals concentrating downstream
- all_chemicals_d11_d12_summary.csv : full breakdown

DEG GENERALITY:
- deg_overlap_heatmap.png : Jaccard between site-dose conditions
- deg_frequency_histogram.png : universal vs specific DEGs

SUMMARY FOR SLIDES:
- 2 distinct pollution regimes along the Danube:
  (a) Agricultural/personal-care chemistry upstream-middle (Diuron, DEET)
  (b) Municipal wastewater downstream (Metformin, sweeteners, drugs)
- Wastewater signature at D11/D12 drives the largest biological response
- D12 also has a unique pesticide pulse (Chlorpropham, Carbaryl, Diazinon,
  Simazine, Chlorophene) — found ONLY at D12
- 29 universal pollution responders cluster into:
  * Lipid metabolism (NPC2, FABP12, LIPC)
  * Phase II detox (SULT1A5)
  * Digestive proteases (suppressed)"

make_zip(
  zipfile = "shared_results.zip",
  files = c(
    "deseq2_results/per_site_10x_summary.csv",
    "deseq2_results/per_site_1x_summary.csv",
    "deseq2_results/degs_per_site.png",
    "deseq2_results/wgcna/module_chemical_heatmap.png",
    "deseq2_results/wgcna/module_dendrogram.png",
    "deseq2_results/wgcna/module_trait_metadata.png",
    "deseq2_results/wgcna/d11_d12_chemistry/top_chems_per_site.png",
    "deseq2_results/wgcna/d11_d12_chemistry/all_chemicals_d11_d12_summary.csv",
    "deseq2_results/wgcna/meblue_deepdive/meblue_eigengene_boxplot.png",
    "deseq2_results/diagnostics/deg_overlap_heatmap.png",
    "deseq2_results/diagnostics/deg_frequency_histogram.png"
  ),
  readme_text = readme_shared
)

cat("\n=== Done ===\n")
cat("Files in", OUTDIR, ":\n")
print(list.files(OUTDIR, full.names = TRUE))
cat("\nSend these to your teammates:\n")
cat("  for_pathway_enrichment.zip  -> pathway teammate\n")
cat("  for_integration.zip         -> multi-omics integration teammate\n")
cat("  shared_results.zip          -> everyone (context)\n")

In [ ]:
###############################################################################
# Reformat sample_sheet.csv into iDEP-compatible format
# iDEP wants:  rows = factors, columns = samples (transposed from typical layout)
###############################################################################

# Load original
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

# Drop excluded samples to match the VST file
ss <- ss[!ss$SampleID %in% c("D03A6", "D03B3"), ]

# Order to match VST column order (defensive — they should already match)
vst_cols <- colnames(read.csv("deseq2_results/wgcna/vst_top5000_for_iDEP.csv",
                              row.names = 1, check.names = FALSE,
                              nrow = 1))
ss <- ss[match(vst_cols, ss$SampleID), ]
stopifnot(all(ss$SampleID == vst_cols))

# Build iDEP-format sample info:
#   Row 1: Treatment (Control / 1x / 10x)
#   Row 2: Site (Control / D01..D12)
# Column names = sample IDs (match VST columns exactly)
idep_si <- data.frame(
  factor = c("Treatment", "Site"),
  rbind(ss$REF, ss$Site),
  stringsAsFactors = FALSE,
  check.names = FALSE
)
colnames(idep_si) <- c("factor", ss$SampleID)

# Save (no quotes, no row names)
write.table(idep_si,
            "sample_sheet_for_iDEP.csv",
            sep = ",", row.names = FALSE, quote = FALSE)

cat("Saved sample_sheet_for_iDEP.csv\n")
cat("Dimensions:", nrow(idep_si), "factor rows x",
    ncol(idep_si) - 1, "sample columns\n")
cat("\nFirst 4 columns preview:\n")
print(idep_si[, 1:5])
cat("\nUpload sample_sheet_for_iDEP.csv as the 'Sample Info file' in iDEP.\n")

## 12. Presentation Figures

In [ ]:
###############################################################################
# PRESENTATION PLOTS — final chunk
# Regenerates clean, slide-ready versions of the key figures from existing CSVs.
# Reads from: deseq2_results/, deseq2_results/wgcna/, deseq2_results/diagnostics/
# Writes to:  deseq2_results/presentation_plots/
#
# Designed for a 3-4 slide presentation:
#   FIG 1 — DEG counts per site (the toxicity ranking)
#   FIG 2 — WGCNA module-chemical heatmap (the headline finding)
#   FIG 3 — The Diuron paradox (Diuron per site vs DEGs per site)
#   FIG 4 — D11/D12 wastewater chemistry
#   FIG 5 — MEblue eigengene across sites/doses (backup/Q&A slide)
###############################################################################
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(pheatmap)
  library(RColorBrewer)
  library(patchwork)
})

OUT <- "deseq2_results/presentation_plots"
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

# Consistent colour scheme across all figures
COLORS <- c(
  "Control" = "#888888",
  "1x"      = "#377eb8",
  "10x"     = "#e41a1c",
  highlight = "#e41a1c",
  neutral   = "#666666"
)

# ============================================================================
# FIG 1 — DEG counts per site (your "toxicity ranking" slide)
# ============================================================================
cat("Building FIG 1: DEGs per site...\n")

degs_10x <- read.csv("deseq2_results/per_site_10x_summary.csv", stringsAsFactors = FALSE)
degs_1x  <- read.csv("deseq2_results/per_site_1x_summary.csv", stringsAsFactors = FALSE)

degs_combined <- bind_rows(
  degs_10x %>% mutate(dose = "10x"),
  degs_1x  %>% mutate(dose = "1x")
)
degs_combined$dose <- factor(degs_combined$dose, levels = c("1x", "10x"))
degs_combined$site <- factor(degs_combined$site,
                             levels = paste0("D", sprintf("%02d", 1:12)))

fig1 <- ggplot(degs_combined, aes(site, n_deg, fill = dose)) +
  geom_col(position = "dodge", colour = "white", linewidth = 0.4) +
  geom_text(aes(label = n_deg), position = position_dodge(width = 0.9),
            vjust = -0.4, size = 3.2, fontface = "bold") +
  scale_fill_manual(values = c("1x" = COLORS["1x"], "10x" = COLORS["10x"]),
                    name = "Concentration") +
  labs(title = "Site-level transcriptional response",
       subtitle = "Number of DEGs per site (padj < 0.05, |log2FC| > 1, vs. Control)",
       x = "Site (upstream  D01  →  D12  downstream)",
       y = "Number of DEGs") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top",
        plot.title = element_text(face = "bold", size = 15),
        axis.text.x = element_text(face = "bold"))

ggsave(file.path(OUT, "FIG1_degs_per_site.png"), fig1,
       width = 11, height = 5.5, dpi = 200)

# ============================================================================
# FIG 2 — WGCNA module-chemical heatmap (the headline finding)
# ============================================================================
cat("Building FIG 2: Module-chemical heatmap...\n")

mod_chem <- read.csv("deseq2_results/wgcna/module_chemical_correlations.csv",
                     stringsAsFactors = FALSE, row.names = 1, check.names = FALSE)

# Pick top 8 modules by max |r| (cleaner than all 17 modules)
abs_max <- apply(abs(as.matrix(mod_chem)), 1, max)
top_mods <- names(sort(abs_max, decreasing = TRUE))[1:8]

# Pick the chemicals where any of these top modules has |r| > 0.25
top_module_data <- as.matrix(mod_chem[top_mods, , drop = FALSE])
chem_max_r <- apply(abs(top_module_data), 2, max)
top_chems <- names(chem_max_r)[chem_max_r > 0.25]

heatmap_mat <- top_module_data[, top_chems, drop = FALSE]

breaks <- seq(-0.6, 0.6, length.out = 101)
colours <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(100)

png(file.path(OUT, "FIG2_module_chemical_heatmap.png"),
    width = 1800, height = 900, res = 150)
pheatmap(heatmap_mat,
         color = colours, breaks = breaks,
         cluster_rows = TRUE, cluster_cols = TRUE,
         show_colnames = TRUE, show_rownames = TRUE,
         fontsize_col = 9, fontsize_row = 11,
         fontsize = 10,
         angle_col = 45,
         main = "Top 8 WGCNA modules vs chemicals (|r| > 0.25)")
dev.off()

# ============================================================================
# FIG 3 — The Diuron paradox (key insight slide)
# Two panels side-by-side: Diuron per site + DEGs per site at 1x
# ============================================================================
cat("Building FIG 3: Diuron paradox panel...\n")

chem <- read.delim("water_chemicals.tsv", stringsAsFactors = FALSE,
                   check.names = FALSE)
rownames(chem) <- chem[[1]]
site_cols_chem <- grep("^D", colnames(chem), value = TRUE)

diuron_row <- grep("^Diuron$", rownames(chem))[1]
diuron_per_site <- as.numeric(chem[diuron_row, site_cols_chem])
names(diuron_per_site) <- site_cols_chem

panel_df <- data.frame(
  site = factor(names(diuron_per_site),
                levels = paste0("D", sprintf("%02d", 1:12))),
  diuron = diuron_per_site
) %>%
  left_join(degs_1x %>% select(site, n_deg), by = "site")

p_diuron <- ggplot(panel_df, aes(site, diuron, fill = diuron > 0)) +
  geom_col(colour = "white", linewidth = 0.4) +
  scale_fill_manual(values = c("TRUE" = COLORS["highlight"],
                               "FALSE" = "#cccccc"),
                    guide = "none") +
  geom_text(aes(label = ifelse(diuron > 0, sprintf("%.1f", diuron), "")),
            vjust = -0.4, size = 3.5, fontface = "bold") +
  labs(title = "Diuron concentration per site",
       subtitle = "Only at 4 sites (D01, D05, D06, D08); zero elsewhere",
       x = NULL, y = "Diuron (\u00b5g/L)") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"))

p_degs <- ggplot(panel_df, aes(site, n_deg, fill = site %in% c("D11", "D12"))) +
  geom_col(colour = "white", linewidth = 0.4) +
  scale_fill_manual(values = c("TRUE" = COLORS["highlight"],
                               "FALSE" = "#999999"),
                    guide = "none") +
  geom_text(aes(label = n_deg), vjust = -0.4, size = 3.5, fontface = "bold") +
  labs(title = "Transcriptional response per site",
       subtitle = "DEG counts at environmental concentration (1x); D11/D12 dominate",
       x = NULL, y = "Number of DEGs") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"))

fig3 <- p_diuron / p_degs +
  plot_annotation(
    title = "The Diuron paradox: chemistry \u2260 biology at the site level",
    subtitle = "Diuron-rich sites are NOT the most transcriptionally affected",
    theme = theme(plot.title = element_text(face = "bold", size = 15))
  )

ggsave(file.path(OUT, "FIG3_diuron_paradox.png"), fig3,
       width = 11, height = 8, dpi = 200)

# ============================================================================
# FIG 4 — D11/D12 wastewater chemistry (the answer to "what's downstream?")
# ============================================================================
cat("Building FIG 4: D11/D12 wastewater chemistry...\n")

# Top 10 chemicals by mean (D11+D12) concentration
chem_mat <- as.matrix(chem[, site_cols_chem])
downstream_mean <- rowMeans(chem_mat[, c("D11", "D12"), drop = FALSE])
top10_dwn <- names(sort(downstream_mean, decreasing = TRUE))[1:10]

chem_long <- chem_mat[top10_dwn, ] %>%
  as.data.frame() %>%
  tibble::rownames_to_column("chemical") %>%
  pivot_longer(-chemical, names_to = "site", values_to = "concentration")

chem_long$site <- factor(chem_long$site,
                         levels = paste0("D", sprintf("%02d", 1:12)))
chem_long$chemical <- factor(chem_long$chemical, levels = top10_dwn)
chem_long$is_downstream <- chem_long$site %in% c("D11", "D12")

fig4 <- ggplot(chem_long, aes(site, concentration, fill = is_downstream)) +
  geom_col() +
  facet_wrap(~ chemical, scales = "free_y", ncol = 5) +
  scale_fill_manual(values = c("TRUE" = COLORS["highlight"],
                               "FALSE" = "#888888"),
                    labels = c("TRUE" = "D11/D12 (downstream)",
                               "FALSE" = "Other sites"),
                    name = NULL) +
  labs(title = "Top 10 chemicals concentrating at D11/D12",
       subtitle = "Wastewater signature: metformin, sweeteners, painkillers, antibiotics",
       x = NULL, y = "Concentration (\u00b5g/L)") +
  theme_minimal(base_size = 11) +
  theme(plot.title = element_text(face = "bold", size = 14),
        strip.text = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
        legend.position = "top")

ggsave(file.path(OUT, "FIG4_d11d12_wastewater.png"), fig4,
       width = 13, height = 8, dpi = 200)

# ============================================================================
# FIG 5 — MEblue eigengene across sites and doses (backup / Q&A)
# ============================================================================
cat("Building FIG 5: MEblue eigengene boxplot...\n")

MEs <- read.csv("deseq2_results/wgcna/module_eigengenes.csv",
                stringsAsFactors = FALSE, row.names = 1)
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)
ss <- ss[!ss$SampleID %in% c("D03A6", "D03B3"), ]

meblue_df <- data.frame(
  SampleID = rownames(MEs),
  MEblue = MEs$MEblue
) %>%
  left_join(ss, by = "SampleID")

meblue_df$REF  <- factor(meblue_df$REF, levels = c("Control", "1x", "10x"))
meblue_df$Site <- factor(meblue_df$Site,
                         levels = c("Control", paste0("D", sprintf("%02d", 1:12))))

fig5 <- ggplot(meblue_df, aes(Site, MEblue, fill = REF)) +
  geom_boxplot(alpha = 0.85, outlier.size = 1.2,
               position = position_dodge(width = 0.85)) +
  geom_hline(yintercept = 0, linetype = "dotted", colour = "grey50") +
  scale_fill_manual(values = c("Control" = COLORS["Control"],
                               "1x" = COLORS["1x"],
                               "10x" = COLORS["10x"]),
                    name = "Concentration") +
  labs(title = "MEblue (Diuron-linked detox module) activity across sites",
       subtitle = "Higher values = stronger detox-module activation",
       x = NULL, y = "MEblue eigengene") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "top")

ggsave(file.path(OUT, "FIG5_meblue_eigengene_boxplot.png"), fig5,
       width = 11, height = 6, dpi = 200)

# ============================================================================
# FIG 6 (bonus) — universal responder summary table as plot
# ============================================================================
cat("Building FIG 6: Universal responders overview...\n")

uni <- read.csv("deseq2_results/diagnostics/universal_responders_anchored.csv",
                stringsAsFactors = FALSE)

# Categorize each gene based on its human symbol
categorise <- function(sym) {
  if (is.na(sym) || sym == "") return("Unmapped")
  if (grepl("^FABP|NPC2|^LIP|^ACSL|^ACSM|GM2A", sym)) return("Lipid metabolism")
  if (grepl("^SULT|ST1A|^GST|^CYP|^UGT|^ABC[BCG]", sym)) return("Phase I/II detox")
  if (grepl("^HSP|^SOD|^CAT$|^GPX|ALDH", sym)) return("Stress / oxidative")
  if (grepl("^PRSS|^KLK|ELANE|TRY", sym)) return("Proteases / digestion")
  if (grepl("^TLR|^IL[0-9]|^NFKB|^CLEC", sym)) return("Immune")
  return("Other characterised")
}

uni$category <- sapply(uni$hsa_symbol, categorise)
cat_counts <- as.data.frame(table(uni$category))
colnames(cat_counts) <- c("category", "count")
cat_counts <- cat_counts %>% arrange(desc(count))
cat_counts$category <- factor(cat_counts$category, levels = cat_counts$category)

fig6 <- ggplot(cat_counts, aes(category, count, fill = category)) +
  geom_col() +
  geom_text(aes(label = count), vjust = -0.4, fontface = "bold", size = 4) +
  scale_fill_brewer(palette = "Set2", guide = "none") +
  labs(title = "Functional categories of 29 universal pollution responders",
       subtitle = "Genes significant in \u226515 of 24 site\u00d7dose conditions",
       x = NULL, y = "Number of genes") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"),
        axis.text.x = element_text(angle = 30, hjust = 1, size = 10))

ggsave(file.path(OUT, "FIG6_universal_responders_categories.png"), fig6,
       width = 9, height = 5.5, dpi = 200)

# ============================================================================
# Final summary
# ============================================================================
cat("\n=== Done ===\n")
cat("Presentation plots saved to:", normalizePath(OUT), "\n\n")

cat("RECOMMENDED SLIDE USE:\n")
cat("  FIG 1 - DEG counts per site         -> Slide 1 (data overview, toxicity ranking)\n")
cat("  FIG 2 - Module-chemical heatmap     -> Slide 2 (the headline: MEblue \u2194 Diuron)\n")
cat("  FIG 3 - Diuron paradox              -> Slide 3 (the killer slide, critical thinking)\n")
cat("  FIG 4 - D11/D12 wastewater          -> Slide 3 OR backup (chemistry context)\n")
cat("  FIG 5 - MEblue eigengene boxplot    -> Backup / Q&A\n")
cat("  FIG 6 - Universal responders table  -> Backup / Q&A\n")

cat("\nFiles produced:\n")
print(list.files(OUT, full.names = FALSE))

In [ ]:
###############################################################################
# PRESENTATION PLOTS — EXTENDED (PATCHED)
# Fix: removed annotation_colors entirely to avoid pheatmap factor-matching error
# Plots will use pheatmap's default auto-colours for annotations (still readable)
###############################################################################

suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(pheatmap)
  library(RColorBrewer)
})

OUT <- "deseq2_results/presentation_plots"
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

COLORS <- c(
  "Control" = "#888888",
  "1x"      = "#377eb8",
  "10x"     = "#e41a1c"
)

# ---- Common loading -------------------------------------------------------
vst <- read.csv("rna_vst_counts.csv", row.names = 1, check.names = FALSE)
ss  <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)

exclude <- c("D03A6", "D03B3")
vst <- vst[, !colnames(vst) %in% exclude]
ss  <- ss[!ss$SampleID %in% exclude, ]
ss  <- ss[match(colnames(vst), ss$SampleID), ]
stopifnot(all(colnames(vst) == ss$SampleID))

ss$REF  <- factor(ss$REF,  levels = c("Control", "1x", "10x"))
ss$Site <- factor(ss$Site, levels = c("Control", paste0("D", sprintf("%02d", 1:12))))

gene_var <- apply(vst, 1, var)
top_idx  <- order(gene_var, decreasing = TRUE)[1:2000]
vst_top  <- as.matrix(vst[top_idx, ])

# ============================================================================
# FIG 7 — PCA (concentration + site, side-by-side)
# ============================================================================
cat("Building FIG 7: PCA...\n")

pca <- prcomp(t(vst_top), scale. = FALSE, center = TRUE)
var_exp <- (pca$sdev^2) / sum(pca$sdev^2) * 100

pca_df <- as.data.frame(pca$x[, 1:3])
pca_df$SampleID <- rownames(pca_df)
pca_df <- pca_df %>% left_join(ss, by = "SampleID")

p_a <- ggplot(pca_df, aes(PC1, PC2, colour = REF)) +
  geom_point(size = 2.6, alpha = 0.85) +
  scale_colour_manual(values = COLORS, name = "Concentration") +
  labs(title = "PCA — coloured by concentration",
       x = sprintf("PC1 (%.1f%%)", var_exp[1]),
       y = sprintf("PC2 (%.1f%%)", var_exp[2])) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"),
        legend.position = "right")

pca_df$site_num <- suppressWarnings(as.numeric(gsub("^D0?", "", as.character(pca_df$Site))))

p_b <- ggplot() +
  geom_point(data = filter(pca_df, is.na(site_num)),
             aes(PC1, PC2),
             colour = "grey60", size = 2.8, shape = 17) +
  geom_point(data = filter(pca_df, !is.na(site_num)),
             aes(PC1, PC2, colour = site_num),
             size = 2.6, alpha = 0.9) +
  scale_colour_viridis_c(option = "viridis",
                         name = "Site\n(upstream\u2192downstream)",
                         breaks = c(1, 4, 8, 12),
                         labels = c("D01", "D04", "D08", "D12")) +
  labs(title = "PCA — coloured by site",
       x = sprintf("PC1 (%.1f%%)", var_exp[1]),
       y = sprintf("PC2 (%.1f%%)", var_exp[2])) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"))

ggsave(file.path(OUT, "FIG7a_pca_concentration.png"), p_a,
       width = 7, height = 5, dpi = 200)
ggsave(file.path(OUT, "FIG7b_pca_site.png"), p_b,
       width = 7, height = 5, dpi = 200)

if (requireNamespace("patchwork", quietly = TRUE)) {
  fig7 <- patchwork::wrap_plots(p_a, p_b, ncol = 2) +
    patchwork::plot_annotation(
      title = "Sample structure: PCA on top 2000 variable genes",
      theme = ggplot2::theme(plot.title = element_text(face = "bold", size = 14))
    )
  ggsave(file.path(OUT, "FIG7_pca_combined.png"), fig7,
         width = 13, height = 5.5, dpi = 200)
}

# ============================================================================
# FIG 8 — Sample correlation heatmap (NO annotation_colors — pheatmap auto-picks)
# ============================================================================
cat("Building FIG 8: Sample correlation heatmap...\n")

cor_mat <- cor(vst_top, method = "pearson")

# Build annotation as plain character vectors (no factors) to avoid the issue
annotation_col <- data.frame(
  Concentration = as.character(ss$REF),
  Site = as.character(ss$Site),
  row.names = ss$SampleID,
  stringsAsFactors = FALSE
)

png(file.path(OUT, "FIG8_sample_correlation_heatmap.png"),
    width = 1500, height = 1300, res = 130)
pheatmap(cor_mat,
         annotation_col = annotation_col,
         show_rownames = FALSE, show_colnames = FALSE,
         color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
         main = "Sample-sample correlation (Pearson, top 2000 variable genes)",
         fontsize = 10)
dev.off()

# ============================================================================
# FIG 9 — Sample dendrogram
# ============================================================================
cat("Building FIG 9: Sample dendrogram...\n")

sample_dist <- as.dist(1 - cor_mat)
sample_tree <- hclust(sample_dist, method = "average")

png(file.path(OUT, "FIG9_sample_dendrogram.png"),
    width = 1600, height = 700, res = 130)
par(mar = c(8, 4, 3, 1))
plot(sample_tree,
     main = "Hierarchical clustering of samples",
     sub = "Distance = 1 - Pearson correlation; Average linkage",
     xlab = "", ylab = "Height",
     cex = 0.5)
dev.off()

# ============================================================================
# FIG 10 — copy existing WGCNA dendrogram if present
# ============================================================================
cat("Building FIG 10: WGCNA gene dendrogram...\n")

orig_dendro <- "deseq2_results/wgcna/module_dendrogram.png"
if (file.exists(orig_dendro)) {
  file.copy(orig_dendro,
            file.path(OUT, "FIG10_wgcna_gene_dendrogram.png"),
            overwrite = TRUE)
  cat("  Copied existing module_dendrogram.png\n")
} else {
  cat("  WARNING: original module_dendrogram.png not found; skipping FIG 10\n")
}

# ============================================================================
# FIG 11 — Module eigengene clustering
# ============================================================================
cat("Building FIG 11: Module eigengene clustering...\n")

MEs <- read.csv("deseq2_results/wgcna/module_eigengenes.csv",
                stringsAsFactors = FALSE, row.names = 1)

ME_dist <- as.dist(1 - cor(MEs))
ME_tree <- hclust(ME_dist, method = "average")

png(file.path(OUT, "FIG11_eigengene_clustering.png"),
    width = 1100, height = 600, res = 130)
par(mar = c(8, 4, 3, 1))
plot(ME_tree,
     main = "Clustering of WGCNA module eigengenes",
     sub = "Distance = 1 - correlation; closer modules behave similarly",
     xlab = "", ylab = "Height",
     cex = 0.9)
abline(h = 0.25, col = "red", lty = 2)
dev.off()

# ============================================================================
# FIG 12 — Top variable genes heatmap (NO annotation_colors)
# ============================================================================
cat("Building FIG 12: Top 500 variable genes heatmap...\n")

top_500 <- order(gene_var, decreasing = TRUE)[1:500]
vst_500 <- as.matrix(vst[top_500, ])
vst_500_z <- t(scale(t(vst_500)))

col_order <- order(ss$REF, ss$Site)
vst_500_z <- vst_500_z[, col_order]
ss_ord    <- ss[col_order, ]

# Plain character annotation — same fix
ann_col <- data.frame(
  Concentration = as.character(ss_ord$REF),
  Site = as.character(ss_ord$Site),
  row.names = ss_ord$SampleID,
  stringsAsFactors = FALSE
)

png(file.path(OUT, "FIG12_top500_genes_heatmap.png"),
    width = 1700, height = 1300, res = 130)
pheatmap(vst_500_z,
         annotation_col = ann_col,
         show_rownames = FALSE, show_colnames = FALSE,
         cluster_cols = FALSE,
         color = colorRampPalette(rev(brewer.pal(11, "RdBu")))(100),
         main = "Top 500 variable genes (z-scored, samples ordered by treatment then site)",
         fontsize = 10)
dev.off()

# ============================================================================
# Summary
# ============================================================================
cat("\n=== Done ===\n")
cat("Plots saved to:", normalizePath(OUT), "\n\n")
print(list.files(OUT, full.names = FALSE))

In [ ]:
# Manually try the FIG 8 plot
suppressPackageStartupMessages({
  library(pheatmap)
  library(RColorBrewer)
  library(dplyr)
})

vst <- read.csv("rna_vst_counts.csv", row.names = 1, check.names = FALSE)
ss  <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)
exclude <- c("D03A6", "D03B3")
vst <- vst[, !colnames(vst) %in% exclude]
ss  <- ss[!ss$SampleID %in% exclude, ]
ss  <- ss[match(colnames(vst), ss$SampleID), ]

gene_var <- apply(vst, 1, var)
top_idx  <- order(gene_var, decreasing = TRUE)[1:2000]
vst_top  <- as.matrix(vst[top_idx, ])

cor_mat <- cor(vst_top, method = "pearson")

# Simplest possible heatmap — no annotations at all
pheatmap(cor_mat,
         show_rownames = FALSE,
         show_colnames = FALSE,
         color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
         main = "Sample-sample correlation",
         filename = "deseq2_results/presentation_plots/FIG8_sample_correlation_heatmap.png",
         width = 12, height = 10)

In [ ]:
ss <- read.csv("sample_sheet.csv", stringsAsFactors = FALSE)
ss <- ss[!ss$SampleID %in% c("D03A6", "D03B3"), ]

# What are the actual unique values?
cat("Unique values in REF:\n")
print(unique(ss$REF))

cat("\nLengths of each:\n")
print(nchar(unique(ss$REF)))   # detects trailing whitespace

In [ ]:
###############################################################################
# Rebuild FIG 2 — module-chemical correlation heatmap
# Uses pheatmap's filename= argument (same approach that worked for FIG 8)
###############################################################################

suppressPackageStartupMessages({
  library(pheatmap)
  library(RColorBrewer)
})

OUT <- "deseq2_results/presentation_plots"
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

# ---- Load module-chemical correlations ------------------------------------
mod_chem <- read.csv("deseq2_results/wgcna/module_chemical_correlations.csv",
                     row.names = 1, check.names = FALSE)

cat("Loaded module-chemical correlations:\n")
cat("  Modules :", nrow(mod_chem), "\n")
cat("  Chemicals:", ncol(mod_chem), "\n")

# Convert explicitly to numeric matrix
mod_chem <- as.matrix(mod_chem)
mode(mod_chem) <- "numeric"

# ---- Filter: top 8 modules by max |r|, chemicals with |r| > 0.25 ----------
abs_max_per_module <- apply(abs(mod_chem), 1, max, na.rm = TRUE)
top_mods <- names(sort(abs_max_per_module, decreasing = TRUE))[1:8]

heatmap_data <- mod_chem[top_mods, , drop = FALSE]

abs_max_per_chem <- apply(abs(heatmap_data), 2, max, na.rm = TRUE)
top_chems <- names(abs_max_per_chem)[abs_max_per_chem > 0.25 &
                                     !is.na(abs_max_per_chem)]

heatmap_mat <- heatmap_data[, top_chems, drop = FALSE]

cat("\nHeatmap dimensions:", nrow(heatmap_mat), "modules x",
    ncol(heatmap_mat), "chemicals\n")
cat("Top modules:", paste(top_mods, collapse = ", "), "\n")
cat("Range of values:", round(range(heatmap_mat, na.rm = TRUE), 3), "\n")

# Replace any NAs with 0 (just in case)
heatmap_mat[is.na(heatmap_mat)] <- 0

# ---- Build heatmap using filename= (the approach that worked for FIG 8) ---
breaks <- seq(-0.6, 0.6, length.out = 101)
colours <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(100)

pheatmap(heatmap_mat,
         color = colours,
         breaks = breaks,
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         show_colnames = TRUE,
         show_rownames = TRUE,
         fontsize_col = 9,
         fontsize_row = 11,
         fontsize = 10,
         angle_col = 45,
         main = "Top 8 WGCNA modules vs chemicals (|r| > 0.25)",
         filename = file.path(OUT, "FIG2_module_chemical_heatmap.png"),
         width = 14,
         height = 7)

cat("\nFIG 2 saved to:", file.path(OUT, "FIG2_module_chemical_heatmap.png"), "\n")
cat("Done.\n")

In [ ]:
modules <- read.csv("deseq2_results/wgcna/module_gene_assignments.csv",
                    stringsAsFactors = FALSE)
cat("Number of unique modules (including grey):",
    length(unique(modules$module_colour)), "\n")
cat("\nModule sizes:\n")
print(sort(table(modules$module_colour), decreasing = TRUE))